<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_02_video_frame_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 2: Video Inventory, Manifest Construction, and Frame Validation

This notebook inventories the physical Kvasir-Capsule videos in persistent storage and links them to the validated metadata produced in Sector 1. It checks file availability and identifier uniqueness, reads technical video properties, and builds a video-level manifest containing file paths, sizes, annotation status, and container metadata.

Current checks cover container opening, first-frame readability, and consistency between the manifest and annotation metadata. Full sequential decoding is being added to count successfully decoded frames, record decoding outcomes, and investigate differences between observed counts and published dataset references.

Video manifests and audit results are saved to persistent storage to support downstream frame extraction, temporal processing, and dataset preparation.

### 1. Install dependencies and imports

In [1]:
%pip install -q \
    pandas \
    numpy \
    pyarrow \
    opencv-python-headless \
    scikit-image \
    scikit-learn \
    iterative-stratification \
    tqdm \
    gdown \
    PyNvVideoCodec==2.2.3 \
    av==18.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 69.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
from collections import defaultdict

import time
import re
import mimetypes
import os
import json
import random
import warnings
import subprocess
import shutil
import sys
import math
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from IPython.display import display
from collections import deque
from uuid import uuid4
from contextlib import contextmanager
from collections import Counter
from itertools import islice
import importlib.metadata as package_metadata
import importlib.util

import filecmp
import lzma
import stat
import tempfile
import cv2
import numpy as np
import pandas as pd
import torch
import zipfile
import tarfile
import zlib
import gzip
import ast
import hashlib
import skimage


from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

import gc
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, CancelledError


In [3]:
# ------------------------------------------------------------------
# Ensure CUDA TorchCodec and Parquet dependencies
# Keeps the installed PyTorch. Reuses the exact CUDA wheel if present.
# ------------------------------------------------------------------


# Explicit CUDA wheels checked against the official compatibility table
# and PyTorch package indexes. CPU-only wheels do not satisfy these pins.
TORCHCODEC_CUDA_PINS = {
    ((2, 10), "12.8"): "0.10.0+cu128",
    ((2, 10), "13.0"): "0.10.0+cu130",
    ((2, 11), "12.8"): "0.11.1+cu128",
    ((2, 11), "13.0"): "0.16.0+cu130",
}


def select_cuda_wheel(torch_version, cuda_version):
    match = re.match(r"^(\d+)\.(\d+)", torch_version)
    if match is None:
        raise RuntimeError(f"Unrecognized PyTorch version: {torch_version!r}")
    if any(marker in torch_version for marker in ("dev", "rc", "a0", "b0")):
        raise RuntimeError("Use a deliberate compatibility choice for a prerelease PyTorch build.")
    release = tuple(map(int, match.groups()))
    target = TORCHCODEC_CUDA_PINS.get((release, cuda_version))
    # TorchCodec >=0.12 documents forward ABI compatibility from torch 2.11.
    if target is None and release >= (2, 11) and cuda_version == "13.0":
        target = "0.16.0+cu130"
    if target is None:
        raise RuntimeError(
            "No explicit CUDA TorchCodec wheel is configured for "
            f"PyTorch {torch_version}, CUDA {cuda_version}. "
            "This setup does not upgrade PyTorch or substitute a CPU wheel. "
            "The parallel alignment cell requires TorchCodec >=0.10. "
            "Check the official compatibility table and CUDA package index."
        )
    return target


if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("Select a CUDA-enabled Colab GPU runtime first.")

_target_codec = select_cuda_wheel(torch.__version__, torch.version.cuda)
_cuda_tag = "cu" + torch.version.cuda.replace(".", "")
_cuda_index = f"https://download.pytorch.org/whl/{_cuda_tag}"

try:
    _installed_codec = package_metadata.version("torchcodec")
except package_metadata.PackageNotFoundError:
    _installed_codec = None

print("PyTorch retained:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("TorchCodec installed:", _installed_codec)
print("TorchCodec required:", _target_codec)

if _installed_codec != _target_codec:
    # Check BEFORE modifying a package whose native libraries may be loaded.
    _codec_loaded = any(
        name == "torchcodec" or name.startswith("torchcodec.")
        for name in sys.modules
    )
    if _codec_loaded:
        raise RuntimeError(
            "Restart the session, then run this setup BEFORE importing TorchCodec. "
            "No packages were changed by this setup."
        )
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-deps", "--only-binary=:all:",
        f"torchcodec=={_target_codec}", "--index-url", _cuda_index,
    ])
    importlib.invalidate_caches()
    if package_metadata.version("torchcodec") != _target_codec:
        raise RuntimeError("The installed TorchCodec version does not match the requested CUDA wheel.")
    print("CUDA TorchCodec installed.")
else:
    print("Matching CUDA TorchCodec already installed; no reinstall.")

if importlib.util.find_spec("pyarrow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])

print("Dependencies ready. The alignment cell will test actual CUDA video decoding.")

PyTorch retained: 2.11.0+cu128
PyTorch CUDA build: 12.8
TorchCodec installed: 0.11.0+cu128
TorchCodec required: 0.11.1+cu128
CUDA TorchCodec installed.
Dependencies ready. The alignment cell will test actual CUDA video decoding.


In [4]:
# ------------------------------------------------------------------
# Load alignment dependencies
# ------------------------------------------------------------------

try:
    import pyarrow
    import torchcodec

    from torchcodec.decoders import (
        VideoDecoder,
        set_cuda_backend,
    )

except (ImportError, RuntimeError, OSError) as error:
    raise RuntimeError(
        "Alignment dependencies could not be imported. "
        "Run the CUDA dependency setup cell before this cell. "
        "If an already imported native package was replaced, "
        "restart the runtime and execute the cells in order."
    ) from error

print("PyTorch:", torch.__version__)
print("TorchCodec:", package_metadata.version("torchcodec"))
print("PyArrow:", pyarrow.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
TorchCodec: 0.11.1+cu128
PyArrow: 23.0.1
CUDA available: True


### 2. Github Colab Sync

In [5]:
GIT_CONFIG = {
    "repo_url": "https://github.com/bogdanparvu18/msc-graduate-project.git",
    "branch": "main",
    "local_repo_dir": "/content/msc-graduate-project",
    "config_notebook_name": "phase2_01_ingestion.ipynb",
    "repo_output_dir": "outputs/phase2",
    "config_output_filename": "phase2_01_ingestion_{run_id}_config.json",
    "output_groups": ["configs", "results", "reports"],
    "dataset_audit_subdir": "dataset_audit",
    "dataset_audit_excluded_artifacts": [
        "physical_image_inventory",
        "metadata_clean",
    ],
    "allowed_extensions": [".json", ".csv", ".md", ".txt"],
    "max_file_size_mib": 20,
    "token_secret_name": "GITHUB_TOKEN",
    "github_username": "bogdanparvu18",
    "author_name": "Bogdan Parvu",
    "author_email": "bparvu@lakeheadu.ca",
    "commit_message": "Phase 2: update ingestion config and audit outputs",
    "confirm_push": True,
}


def run_git(repo, *args, env=None, input_text=None, allowed_codes=(0,)):
    """Runs Git without a shell; never embeds credentials in arguments."""
    command = ["git"] + (["-C", str(repo)] if repo is not None else [])
    process_env = {
        k: v for k, v in os.environ.items()
        if not k.startswith("GIT_TRACE") and k != "GIT_CURL_VERBOSE"
    }
    process_env.update({"GIT_TERMINAL_PROMPT": "0", **(env or {})})
    result = subprocess.run(
        command + list(args), input=input_text, env=process_env,
        text=True, capture_output=True,
    )
    if result.returncode not in allowed_codes:
        message = (result.stderr or result.stdout).strip()
        token = process_env.get("GITHUB_TOKEN")
        if token:
            message = message.replace(token, "[REDACTED]")
        raise RuntimeError(f"Git failed:\n{message}")
    return result.stdout


def prepare_git_repository(settings):
    """Clones once or fast-forwards a clean, matching local repository."""
    repo = Path(settings["local_repo_dir"]).expanduser().resolve()
    branch = settings["branch"]
    if not (repo / ".git").is_dir():
        if repo.exists() and (not repo.is_dir() or any(repo.iterdir())):
            raise RuntimeError(f"Destination exists but is not an empty Git checkout: {repo}")
        repo.parent.mkdir(parents=True, exist_ok=True)
        run_git(None, "clone", "--branch", branch, "--single-branch",
                settings["repo_url"], str(repo))

    expected = settings["repo_url"].rstrip("/").removesuffix(".git")
    for options in [("get-url",), ("get-url", "--push")]:
        actual = run_git(repo, "remote", *options, "origin").strip()
        if actual.rstrip("/").removesuffix(".git") != expected:
            raise RuntimeError("Origin does not match GIT_CONFIG. Nothing was pushed.")
    if run_git(repo, "branch", "--show-current").strip() != branch:
        raise RuntimeError(f"Expected branch {branch}; no automatic branch switching.")
    if run_git(repo, "status", "--porcelain").strip():
        raise RuntimeError("Local checkout has changes. Resolve them before synchronizing; no reset is performed.")

    run_git(repo, "pull", "--ff-only", "origin", branch)
    if run_git(repo, "rev-list", f"origin/{branch}..HEAD").strip():
        raise RuntimeError("There are unpublished local commits. Resolve/push them before starting another sync.")
    return repo


def load_config_from_notebook(repo, notebook_name):
    """Reads the literal CONFIG dictionary; does NOT execute the notebook."""
    tracked = run_git(repo, "ls-files", "-z").split("\0")
    matches = [repo / p for p in tracked if p and Path(p).name == notebook_name]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one tracked {notebook_name}; found: {matches}")
    notebook_path = matches[0]
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    configs = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        source = "".join(source) if isinstance(source, list) else source
        try:
            statements = ast.parse(source).body
        except SyntaxError:
            if re.search(r"(?m)^\s*CONFIG\s*=", source):
                raise ValueError("Keep CONFIG = {...} in a plain Python cell without shell/magic commands.") from None
            continue
        for node in statements:
            targets = node.targets if isinstance(node, ast.Assign) else (
                [node.target] if isinstance(node, ast.AnnAssign) else []
            )
            if not any(isinstance(t, ast.Name) and t.id == "CONFIG" for t in targets):
                continue
            if not isinstance(node.value, ast.Dict):
                continue  # Ignore calls such as CONFIG = load_config(...).
            try:
                configs.append(ast.literal_eval(node.value))
            except (ValueError, TypeError, SyntaxError):
                raise ValueError("CONFIG must contain literal values, not calls, external variables, or **CONFIG.") from None
    if len(configs) != 1:
        raise ValueError(f"Expected one literal CONFIG dictionary; found {len(configs)}.")
    json.dumps(configs[0], allow_nan=False)  # Check JSON compatibility.
    return configs[0], notebook_path


def file_sha256(path):
    """Hashes file contents without loading the entire file into RAM."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def collect_publishable_outputs(dirs, repo, settings):
    """Builds a publication plan from output directories only."""
    relative_root = Path(settings["repo_output_dir"])
    if relative_root.is_absolute() or ".." in relative_root.parts:
        raise ValueError("repo_output_dir must be a repository-relative path.")
    records, skipped = [], []
    for group in settings["output_groups"]:
        if group not in {"configs", "results", "reports"}:
            raise ValueError(f"Unsupported output group: {group}")
        source_dir = Path(dirs[f"{group}_dir"]).resolve()
        if not source_dir.is_dir():
            raise FileNotFoundError(f"Output directory is missing: {source_dir}")
        for source in sorted(source_dir.rglob("*")):
            if not source.is_file():
                continue
            relative = source.relative_to(source_dir)
            if any(part.startswith(".") for part in relative.parts) or source.suffix.lower() not in settings["allowed_extensions"]:
                skipped.append(f"{group}/{relative.as_posix()}")
                continue
            if source.is_symlink() or not source.resolve().is_relative_to(source_dir):
                raise ValueError(f"Refusing a symlink/out-of-scope source: {source}")
            size = source.stat().st_size
            if size > settings["max_file_size_mib"] * 1024 ** 2:
                raise ValueError(f"File exceeds publication size limit: {source}")
            repo_relative = relative_root / group / relative
            destination = repo / repo_relative
            if destination.is_symlink() or not destination.resolve().is_relative_to(repo):
                raise ValueError(f"Unsafe destination: {destination}")
            digest = file_sha256(source)
            records.append({
                "source": source, "destination": destination,
                "repo_relative": repo_relative.as_posix(), "group": group,
                "size": size, "sha256": digest,
                "changed": not destination.exists() or file_sha256(destination) != digest,
            })
    return records, skipped


def build_dataset_audit_github_files(
    storage_spec, catalog_filename, excluded_artifacts,
):
    """Derives audit filenames from the specification used by the exporter."""
    missing_columns = sorted({"artifact", "format"} - set(storage_spec.columns))
    if missing_columns:
        raise KeyError(f"Audit storage specification is missing: {missing_columns}")

    selected = (
        storage_spec.loc[
            ~storage_spec["artifact"].isin(excluded_artifacts),
            ["artifact", "format"],
        ]
        .astype("string")
        .assign(filename=lambda data: data["artifact"] + "." + data["format"])
    )
    # The exporter writes this catalog separately from AUDIT_STORAGE_SPEC.
    filenames = pd.concat(
        [selected["filename"], pd.Series([catalog_filename], dtype="string")],
        ignore_index=True,
    )
    valid = filenames.str.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", na=False)
    if not valid.all():
        raise ValueError(
            "Audit exports must have nonempty, simple filenames: "
            f"{filenames.loc[~valid].tolist()}"
        )
    return tuple(filenames.drop_duplicates().sort_values())


def collect_publishable_dataset_audit(
    dirs, repo, settings, storage_spec, catalog_filename,
):
    """Plans audit publication; existing remote files are never updated.

    Call after prepare_git_repository(), so HEAD matches the remote branch.
    This function inspects files but does not copy, stage, commit or push.
    """
    subdir = Path(settings["dataset_audit_subdir"])
    if subdir.is_absolute() or ".." in subdir.parts or subdir == Path("."):
        raise ValueError("dataset_audit_subdir must be a nonempty relative path.")
    relative_root = Path(settings["repo_output_dir"])
    if relative_root.is_absolute() or ".." in relative_root.parts:
        raise ValueError("repo_output_dir must be a repository-relative path.")

    source_dir = (Path(dirs["curated_data_dir"]) / subdir).resolve()
    destination_root = relative_root / subdir
    excluded_artifacts = settings["dataset_audit_excluded_artifacts"]
    filenames = build_dataset_audit_github_files(
        storage_spec, catalog_filename, excluded_artifacts,
    )
    excluded_spec = storage_spec.loc[
        storage_spec["artifact"].isin(excluded_artifacts), ["artifact", "format"]
    ].astype("string")
    skipped = [
        f"{subdir.as_posix()}/{name}"
        for name in (excluded_spec["artifact"] + "." + excluded_spec["format"])
    ]
    allowed_extensions = {extension.lower() for extension in settings["allowed_extensions"]}
    tracked_files = set(filter(None, run_git(
        repo, "--literal-pathspecs", "ls-tree", "-r", "--name-only", "-z",
        "HEAD", "--", destination_root.as_posix(),
    ).split("\0")))

    records = []
    for filename in filenames:
        if Path(filename).suffix.lower() not in allowed_extensions:
            skipped.append(f"{subdir.as_posix()}/{filename}")
            continue
        source = source_dir / filename
        repo_relative = (destination_root / filename).as_posix()
        destination = repo / repo_relative
        if destination.is_symlink() or not destination.resolve().is_relative_to(repo):
            raise ValueError(f"Unsafe destination: {destination}")

        already_on_github = repo_relative in tracked_files
        size, digest = None, None
        if not already_on_github:
            if not source.is_file():
                raise FileNotFoundError(
                    f"Audit report is absent from GitHub and Drive: {source}. "
                    "Run the dataset audit export cell first."
                )
            if source.is_symlink() or not source.resolve().is_relative_to(source_dir):
                raise ValueError(f"Refusing a symlink/out-of-scope source: {source}")
            size = source.stat().st_size
            if size > settings["max_file_size_mib"] * 1024 ** 2:
                raise ValueError(f"File exceeds publication size limit: {source}")
            digest = file_sha256(source)

        records.append({
            "source": source, "destination": destination,
            "repo_relative": repo_relative, "group": "dataset_audit",
            "size": size, "sha256": digest,
            "changed": not already_on_github,
        })
    return records, skipped


def authenticated_push(repo, settings, token):
    """Passes the token via an ephemeral environment, not the remote URL."""
    with tempfile.TemporaryDirectory(prefix="mmvqa-git-") as folder:
        askpass = Path(folder) / "askpass.sh"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '*Username*) printf "%s\\n" "$MMVQA_GIT_USER" ;;\n'
            '*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            'esac\n', encoding="utf-8",
        )
        askpass.chmod(0o700)
        environment = {
            "GIT_ASKPASS": str(askpass),
            "MMVQA_GIT_USER": settings["github_username"],
            "GITHUB_TOKEN": token,
            "LC_ALL": "C",
        }
        run_git(repo, "-c", "credential.helper=", "push", "origin",
                f"HEAD:refs/heads/{settings['branch']}", env=environment)


def publish_phase2_outputs(config, dirs, settings):
    """Publishes runtime CONFIG, saved outputs and missing dataset audit files.

    Regular outputs are updated when their content changes. Audit filenames
    follow AUDIT_STORAGE_SPEC; audit files already on GitHub are preserved.
    All selected changes share the existing confirmation, commit and push.
    """
    if config["storage_backend"] == "google_drive" and not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Google Drive before publishing.")
    expected_root = Path(config["output_dir"])
    if not expected_root.is_absolute():
        expected_root = Path(config["storage_root"]) / expected_root
    expected_root = expected_root.resolve()
    for group in settings["output_groups"]:
        if Path(dirs[f"{group}_dir"]).resolve() != expected_root / group:
            raise ValueError("DIRS and CONFIG point to different output locations.")
        if not Path(dirs[f"{group}_dir"]).is_dir():
            raise FileNotFoundError(f"Run prepare_phase2_dirs first: {group}")

    # Audit definitions are needed only here, not during the bootstrap below.
    missing_audit_definitions = [
        name for name in ("AUDIT_STORAGE_SPEC", "AUDIT_CATALOG_FILENAME")
        if name not in globals()
    ]
    if missing_audit_definitions:
        raise RuntimeError(
            "Run the dataset audit definition/export cell before publishing. "
            f"Missing: {missing_audit_definitions}"
        )

    repo = prepare_git_repository(settings)  # Refresh before copying, not afterwards.
    run_id = config.get("run_id")
    if not isinstance(run_id, str) or not run_id.strip():
        raise ValueError("Initialize CONFIG['run_id'] at the start of the run before publishing.")
    config_filename = settings["config_output_filename"].format(run_id=run_id)
    if Path(config_filename).name != config_filename or not config_filename.endswith(".json"):
        raise ValueError("config_output_filename must resolve to a JSON filename, not a path.")
    config_path = Path(dirs["configs_dir"]) / config_filename
    config_text = json.dumps(config, indent=2, ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n"
    config_path.write_text(config_text, encoding="utf-8")
    records, skipped = collect_publishable_outputs(dirs, repo, settings)
    audit_records, audit_skipped = collect_publishable_dataset_audit(
        dirs=dirs, repo=repo, settings=settings,
        storage_spec=AUDIT_STORAGE_SPEC,
        catalog_filename=AUDIT_CATALOG_FILENAME,
    )
    records = [*records, *audit_records]
    skipped = [*skipped, *audit_skipped]
    planned_paths = pd.Series([r["repo_relative"] for r in records], dtype="string")
    if planned_paths.duplicated().any():
        raise ValueError("Multiple publication sources target the same repository path.")
    if not any(r["group"] in {"reports", "results", "dataset_audit"} for r in records):
        raise RuntimeError("No publishable reports/results found. Save them to Drive first.")
    audit_publication = {
        "already_on_github": [r["repo_relative"] for r in audit_records if not r["changed"]],
        "published": [],
    }
    changed = [r for r in records if r["changed"]]
    print(f"Eligible: {len(records)} | Changed: {len(changed)} | Excluded: {len(skipped)}")
    print(f"Dataset audit already on GitHub: {len(audit_publication['already_on_github'])}")
    for path in audit_publication["already_on_github"]:
        print("ALREADY ON GITHUB:", path)
    for path in skipped:
        print("EXCLUDED:", path)
    if not changed:
        print("No files need publication under the configured policies. No commit or push is needed.")
        return {"status": "unchanged", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}

    paths = [r["repo_relative"] for r in changed]
    path_input = "\0".join(paths) + "\0"
    ignored = run_git(repo, "check-ignore", "--stdin", "-z",
                      input_text=path_input, allowed_codes=(0, 1))
    if ignored:
        raise RuntimeError("Adjust .gitignore for these outputs first: " + ", ".join(filter(None, ignored.split("\0"))))
    for record in changed:
        print(f"PUBLISH: {record['repo_relative']} ({record['size'] / 1024:.1f} KiB)")
    if settings["confirm_push"] and input("Publish these files? Type PUSH: ").strip() != "PUSH":
        return {"status": "cancelled", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}

    from google.colab import userdata
    token = userdata.get(settings["token_secret_name"]).strip()
    if not token:
        raise ValueError("The GitHub token is empty.")
    email = settings["author_email"].strip() or input("Git commit email (GitHub/noreply): ").strip()
    if "@" not in email:
        raise ValueError("Provide your GitHub commit email or your exact GitHub noreply address.")
    # Detect changes made while the user was reviewing the publication plan.
    if any(file_sha256(r["source"]) != r["sha256"] for r in changed):
        raise RuntimeError("An output changed during review. Run publication again.")
    if any(token.encode() in r["source"].read_bytes() for r in changed):
        raise ValueError("An output contains the GitHub token. Publication stopped.")
    for record in changed:
        record["destination"].parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(record["source"], record["destination"])
    run_git(repo, "--literal-pathspecs", "add", "--pathspec-from-file=-",
            "--pathspec-file-nul", input_text=path_input)
    staged = set(filter(None, run_git(repo, "diff", "--cached", "--name-only", "-z").split("\0")))
    if staged - set(paths):
        raise RuntimeError("Unexpected staged files. Inspect the local checkout before proceeding.")
    if not staged:
        print("No changes remain after Git text normalization.")
        return {"status": "unchanged", "published": 0, "excluded": skipped,
                "dataset_audit": audit_publication}
    print(run_git(repo, "diff", "--cached", "--stat"))
    run_git(repo, "-c", f"user.name={settings['author_name']}", "-c", f"user.email={email}",
            "commit", "-m", settings["commit_message"])
    authenticated_push(repo, settings, token)
    commit = run_git(repo, "rev-parse", "HEAD").strip()
    audit_publication["published"] = sorted(
        staged & {r["repo_relative"] for r in audit_records}
    )
    print(f"Pushed {len(staged)} files to {settings['branch']}. Commit: {commit}")
    return {"status": "pushed", "published": len(staged), "commit": commit,
            "excluded": skipped, "dataset_audit": audit_publication}


# Bootstrap only: no dataset download and no report generation.
REPO_DIR = prepare_git_repository(GIT_CONFIG)

CONFIG, CONFIG_NOTEBOOK_PATH = load_config_from_notebook(
    repo=REPO_DIR,
    notebook_name=GIT_CONFIG["config_notebook_name"],
)

# Initialize ONCE, before running the pipeline, not inside publication.
# Literal defaults maintain compatibility with existing repository CONFIGs.
# Defining these two keys in the source CONFIG overrides these defaults.
CONFIG.setdefault("timezone", "America/Toronto")
CONFIG.setdefault("run_id_format", "%Y%m%d_%H%M%S")

RUN_ID = datetime.now(
    ZoneInfo(CONFIG["timezone"])
).strftime(CONFIG["run_id_format"])

CONFIG["run_id"] = RUN_ID

CONFIG_SOURCE_COMMIT = run_git(REPO_DIR, "rev-parse", "HEAD").strip()

print("Repository:", REPO_DIR)
print("CONFIG loaded from:", CONFIG_NOTEBOOK_PATH.relative_to(REPO_DIR))
print("Source commit:", CONFIG_SOURCE_COMMIT)
print("Run ID:", CONFIG["run_id"])

Repository: /content/msc-graduate-project
CONFIG loaded from: notebooks/phase2/phase2_01_ingestion.ipynb
Source commit: 183cd1c00885b97b324d97f674af54702e08680b
Run ID: 20260925_211052


### 3. Load declarative configuration and reproducibility

In [6]:
CONFIG = {
    # --------------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------------

    "seed": 42,

    # --------------------------------------------------------------
    # Persistent storage
    # --------------------------------------------------------------

    "storage_backend": "google_drive",

    "storage_root": (
        "/content/drive/MyDrive/"
        "MMVQA_Clinical"
    ),

    "raw_data_dir": (
        "data/raw/kvasir_capsule"
    ),

    "interim_data_dir": (
        "data/interim/phase2"
    ),

    "curated_data_dir": (
        "data/curated/phase2"
    ),

    "output_dir": (
        "outputs/phase2"
    ),

    "results_dir": (
        "outputs/phase2/results/"
    ),

    # --------------------------------------------------------------
    # Raw-dataset acquisition
    # --------------------------------------------------------------

    "dataset_download_enabled": True,

    "dataset_google_drive_folder_url": (
        "https://drive.google.com/drive/folders/"
        "18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z"
    ),

    "dataset_archive_repair_enabled": True,

    # --------------------------------------------------------------
    # Raw-dataset validation
    # --------------------------------------------------------------

    "dataset_validation": {
        "osf_storage_subdir": (
            "osfstorage"
        ),

        "required_metadata_file": (
            "metadata.csv"
        ),

        "labelled_images_subdir": (
            "labelled_images"
        ),

        "minimum_video_files": 117,

        "minimum_labelled_images": 47238,

        "image_extensions": [
            ".png",
            ".jpg",
            ".jpeg",
        ],

        "video_extensions": [
            ".avi",
            ".mp4",
            ".mkv",
        ],
    },


    # --------------------------------------------------------------
    # Normalized clinical taxonomy
    # --------------------------------------------------------------

    # Keys must match finding_class_normalized.
    "clinical_group_map": {
        "ampulla of vater": (
            "anatomical_landmark"
        ),

        "ileocecal valve": (
            "anatomical_landmark"
        ),

        "pylorus": (
            "anatomical_landmark"
        ),

        "normal clean mucosa": (
            "normal_mucosa"
        ),

        "reduced mucosal view": (
            "visibility_limitation"
        ),

        "blood fresh": "bleeding",

        "blood hematin": "bleeding",

        "angiectasia": (
            "vascular_lesion"
        ),

        "erosion": (
            "mucosal_lesion"
        ),

        "erythema": (
            "mucosal_lesion"
        ),

        "ulcer": (
            "mucosal_lesion"
        ),

        "lymphangiectasia": (
            "lymphatic_lesion"
        ),

        "polyp": (
            "protruding_lesion"
        ),

        "foreign body": (
            "foreign_body"
        ),
    },

    # --------------------------------------------------------------
    # Video and frame alignment
    # --------------------------------------------------------------


        "expected_export_container_fps": 30.0,

        "frame_index_offset_candidates": [
            -1,
            0,
            1,
        ],

        "frame_alignment_min_ssim": 0.99,
        "frame_alignment_min_margin": 0.005,

    # --------------------------------------------------------------
    # CUDA frame alignment
    # --------------------------------------------------------------
        "frame_alignment_cuda_device": "cuda:0",
        "frame_alignment_cuda_backend": "auto",
        "frame_alignment_video_workers": "all",
        "frame_alignment_mapping_batch_size": 8,
        "frame_alignment_ssim_batch_size": 8,
        "frame_alignment_image_workers": 8,
        "frame_alignment_copy_workers": 4,
        "frame_alignment_vram_budget_fraction": 0.55,
        "frame_alignment_stage_video_locally": True,
        "frame_alignment_local_work_dir": "/content/frame_alignment_cuda_work",
        "frame_alignment_local_disk_reserve_gib": 2.0,
        "frame_alignment_force_rescore": False,
        "frame_alignment_hash_source_bytes": False,
        "frame_alignment_show_progress": True,

    # --------------------------------------------------------------
    # Expected dataset characteristics
    # --------------------------------------------------------------

        "expected_labelled_frames": 47238,
        "expected_classes": 14,

        "expected_labelled_videos": 43,

        "expected_unlabelled_videos": 74,

        "expected_total_videos": 117,

        "expected_total_extractable_frames": (
            4741504
        ),

        "expected_unlabelled_frames": (
            4694266
        ),


    # --------------------------------------------------------------
    # Video decode audit
    # --------------------------------------------------------------

        "decode_audit_backend": "nvdec",
        "decode_audit_gpu_id": 0,
        "decode_audit_allow_cpu_fallback": True,
        "decode_audit_cpu_threads": 2,

        "decode_audit_stage_video_locally": True,
        "decode_audit_local_work_dir": "/content/video_decode_audit_work",

        "decode_audit_force_redecode": False,
        "decode_audit_retry_failed": False,
        "decode_audit_retry_video_keys": [],

        "decode_audit_published_reference_frames": {
            "partially_labelled": 1_955_675,
            "fully_unlabelled": 2_785_829,
            "all_videos": 4_741_504,
        },


}

CONFIG.setdefault("timezone", "America/Toronto")
CONFIG["run_id"] = datetime.now(
    ZoneInfo(CONFIG["timezone"])
).strftime("%Y%m%d_%H%M%S")

CONFIG

{'seed': 42,
 'storage_backend': 'google_drive',
 'storage_root': '/content/drive/MyDrive/MMVQA_Clinical',
 'raw_data_dir': 'data/raw/kvasir_capsule',
 'interim_data_dir': 'data/interim/phase2',
 'curated_data_dir': 'data/curated/phase2',
 'output_dir': 'outputs/phase2',
 'results_dir': 'outputs/phase2/results/',
 'dataset_download_enabled': True,
 'dataset_google_drive_folder_url': 'https://drive.google.com/drive/folders/18vEHN1CG7oNFKdT2NmhtJjrFhb3tLG1Z',
 'dataset_archive_repair_enabled': True,
 'dataset_validation': {'osf_storage_subdir': 'osfstorage',
  'required_metadata_file': 'metadata.csv',
  'labelled_images_subdir': 'labelled_images',
  'minimum_video_files': 117,
  'minimum_labelled_images': 47238,
  'image_extensions': ['.png', '.jpg', '.jpeg'],
  'video_extensions': ['.avi', '.mp4', '.mkv']},
 'clinical_group_map': {'ampulla of vater': 'anatomical_landmark',
  'ileocecal valve': 'anatomical_landmark',
  'pylorus': 'anatomical_landmark',
  'normal clean mucosa': 'normal_mu

### 4. Mount Google Drive Storage Backend

In [7]:
def mount_storage(config):
    """
    Mounts persistent storage when required by the configured backend.

    For Google Colab + Google Drive, this mounts Drive under /content/drive.
    It does not download or copy the dataset.
    """

    storage_backend = config["storage_backend"]

    if storage_backend == "google_drive":

        try:
            from google.colab import drive

            drive.mount(
                "/content/drive",
                force_remount=False,
            )

            print("Google Drive mounted.")

        except ImportError:
            raise RuntimeError(
                "Google Drive backend is configured, "
                "but the notebook is not running in Google Colab."
            )

    elif storage_backend == "local":

        print("Using local storage.")

    else:

        raise ValueError(
            f"Unsupported storage backend: {storage_backend}"
        )


mount_storage(CONFIG)

Mounted at /content/drive
Google Drive mounted.


### 5. Define data paths

In [8]:
def prepare_phase2_dirs(config):
    """
    Resolves and creates the directory structure required
    for Phase 2.

    Main directories are declared in CONFIG. Relative paths
    are resolved against storage_root, while absolute paths
    are preserved.

    Raw-dataset reference paths are returned, but dataset
    source files and source subdirectories are not created.

    Side effect:
        Creates writable Phase 2 directories on persistent
        storage.

    Returns:
        Dictionary containing resolved Path objects.
    """

    storage_root = Path(
        config["storage_root"]
    )

    validation = config[
        "dataset_validation"
    ]


    def resolve_path(path_value):
        """
        Resolves a CONFIG path against storage_root.

        Absolute paths are returned unchanged.
        """

        path = Path(path_value)

        if path.is_absolute():
            return path

        return storage_root / path


    # --------------------------------------------------------------
    # Main CONFIG directories
    # --------------------------------------------------------------

    raw_data_dir = resolve_path(
        config["raw_data_dir"]
    )

    interim_data_dir = resolve_path(
        config["interim_data_dir"]
    )

    curated_data_dir = resolve_path(
        config["curated_data_dir"]
    )

    output_dir = resolve_path(
        config["output_dir"]
    )


    # --------------------------------------------------------------
    # Raw-dataset source references
    # --------------------------------------------------------------

    dataset_root_dir = (
        raw_data_dir
        / validation["osf_storage_subdir"]
    )

    labelled_images_dir = (
        dataset_root_dir
        / validation["labelled_images_subdir"]
    )

    metadata_path = (
        dataset_root_dir
        / validation["required_metadata_file"]
    )


    # --------------------------------------------------------------
    # Writable directories created by the pipeline
    # --------------------------------------------------------------

    created_directories = {
        # Main directories
        "raw_data_dir":
            raw_data_dir,

        "interim_data_dir":
            interim_data_dir,

        "curated_data_dir":
            curated_data_dir,

        "output_dir":
            output_dir,

        # Derived data directories
        "temporal_frames_dir":
            interim_data_dir
            / "temporal_frames",

        "manifests_dir":
            curated_data_dir
            / "manifests",

        "splits_dir":
            curated_data_dir
            / "splits",

        # Derived output directories
        "configs_dir":
            output_dir
            / "configs",

        "results_dir":
            output_dir
            / "results",

        "reports_dir":
            output_dir
            / "reports",
    }


    # --------------------------------------------------------------
    # Create only writable pipeline directories
    # --------------------------------------------------------------

    for directory in created_directories.values():
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )


    # --------------------------------------------------------------
    # Dataset paths that must come from the OSF dataset
    # --------------------------------------------------------------

    dataset_paths = {
        "dataset_root_dir":
            dataset_root_dir,

        "labelled_images_dir":
            labelled_images_dir,

        "metadata_path":
            metadata_path,
    }


    return {
        **created_directories,
        **dataset_paths,
    }


DIRS = prepare_phase2_dirs(
    CONFIG
)

DIRS


config_path = (DIRS["configs_dir"]) / (
    f"phase2_02_video_frame_validation_{CONFIG['run_id']}_config.json"
)
config_path.parent.mkdir(parents=True, exist_ok=True)

config_path.write_text(
    json.dumps(
        CONFIG,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
        allow_nan=False,
    ) + "\n",
    encoding="utf-8",
)

print("Phase 2 Sector 2 configuration saved to:", config_path)

Phase 2 Sector 2 configuration saved to: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/configs/phase2_02_video_frame_validation_20260925_211052_config.json


### 6. Verify input paths and inspect video files and metadata columns

In [9]:
root = DIRS["dataset_root_dir"]
metadata_path = DIRS["metadata_path"]

if not root.is_dir():
    raise FileNotFoundError(f"Dataset root is missing: {root}")

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata is missing: {metadata_path}")

print("Dataset root:", root)
print("\nTop-level entries:")
for path in islice(root.iterdir(), 30):
    print("DIR " if path.is_dir() else "FILE", path.name)

# Sector 1 reads this metadata as semicolon-delimited.
metadata_preview = pd.read_csv(metadata_path, sep=";", nrows=30)
print("\nMetadata columns:", metadata_preview.columns.tolist())
display(metadata_preview)

video_extensions = {
    ext.lower()
    for ext in CONFIG["dataset_validation"]["video_extensions"]
}
video_files = sorted(
    path
    for path in root.rglob("*")
    if path.suffix.lower() in video_extensions
    and path.is_file()
)

print("\nVideos found:", len(video_files))
print("Video directories:")
for directory, count in Counter(
    str(path.parent.relative_to(root)) for path in video_files
).most_common(10):
    print(f"  {directory}: {count}")

print("\nSample video paths:")
for path in video_files[:10]:
    print(" ", path.relative_to(root))

Dataset root: /content/drive/MyDrive/MMVQA_Clinical/data/raw/kvasir_capsule/osfstorage

Top-level entries:
FILE metadata.json
FILE metadata.csv
DIR  labelled_images
DIR  labelled_videos
DIR  unlabelled_videos

Metadata columns: ['filename', 'video_id', 'frame_number', 'finding_category', 'finding_class', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4']


,filename,video_id,frame_number,finding_category,finding_class,x1,y1,x2,y2,x3,y3,x4,y4
0,0728084c8da942d9_22803.jpg,0728084c8da942d9,22803,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0728084c8da942d9_22804.jpg,0728084c8da942d9,22804,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0728084c8da942d9_22805.jpg,0728084c8da942d9,22805,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0728084c8da942d9_22806.jpg,0728084c8da942d9,22806,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0728084c8da942d9_22807.jpg,0728084c8da942d9,22807,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0728084c8da942d9_22808.jpg,0728084c8da942d9,22808,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0728084c8da942d9_22809.jpg,0728084c8da942d9,22809,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0728084c8da942d9_22810.jpg,0728084c8da942d9,22810,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0728084c8da942d9_22811.jpg,0728084c8da942d9,22811,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0728084c8da942d9_22812.jpg,0728084c8da942d9,22812,Luminal,Normal clean mucosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Videos found: 117
Video directories:
  unlabelled_videos: 74
  labelled_videos: 43

Sample video paths:
  labelled_videos/04a78ef00c5245e0.mp4
  labelled_videos/0531325b64674948.mp4
  labelled_videos/0728084c8da942d9.mp4
  labelled_videos/07c1fa15a20a4398.mp4
  labelled_videos/131368cc17e44240.mp4
  labelled_videos/2fc3db471f9d44c0.mp4
  labelled_videos/39960e5e099a45ca.mp4
  labelled_videos/3ada4222967f421d.mp4
  labelled_videos/3c8d5f0b90d7475d.mp4
  labelled_videos/4560e83f9afc4685.mp4


### 7. Load validated metadata from Sector 1

In [10]:
metadata_path = (
    DIRS["curated_data_dir"]
    / "dataset_audit"
    / "metadata_clean.parquet"
)

if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)

df_clean = pd.read_parquet(metadata_path)
display(df_clean[["video_id", "video_key", "frame_number"]].head())

,video_id,video_key,frame_number
0,0728084c8da942d9,0728084c8da942d9,22803
1,0728084c8da942d9,0728084c8da942d9,22804
2,0728084c8da942d9,0728084c8da942d9,22805
3,0728084c8da942d9,0728084c8da942d9,22806
4,0728084c8da942d9,0728084c8da942d9,22807


### 8. Match annotated video IDs to physical video files

In [11]:
video_index = {}

for path in video_files:
    key = path.stem

    if key in video_index:
        raise ValueError(f"Duplicate video filename stem: {key}")

    video_index[key] = path

annotated_keys = set(df_clean["video_key"].dropna())
missing_keys = annotated_keys - set(video_index)

video_inventory = pd.DataFrame([
    {
        "video_id_from_filename": path.stem,
        "relative_path": path.relative_to(root).as_posix(),
        "filename": path.name,
        "size_bytes": path.stat().st_size,
    }
    for path in video_files
])

minimum = CONFIG["dataset_validation"]["minimum_video_files"]
if len(video_inventory) < minimum:
    raise ValueError(
        f"Found {len(video_inventory)} videos; expected at least {minimum}"
    )

inventory_path = DIRS["manifests_dir"] / "video_inventory.csv"
video_inventory.to_csv(inventory_path, index=False)

print(f"Saved {len(video_inventory)} videos to {inventory_path}")

print("Annotated video IDs:", len(annotated_keys))
print("IDs without a matching file:", len(missing_keys))
print("Examples:", sorted(missing_keys)[:10])

Saved 117 videos to /content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests/video_inventory.csv
Annotated video IDs: 43
IDs without a matching file: 0
Examples: []


### 9. Build physical video manifest

In [12]:
# ------------------------------------------------------------------
# Probe video metadata
# ------------------------------------------------------------------

VIDEO_PROBE_COLUMNS = [
    "container_opened",
    "first_frame_readable",
    "container_fps",
    "container_reported_frame_count",
    "width",
    "height",
    "estimated_container_duration_seconds",
]


def empty_video_probe():
    """
    Returns the stable technical schema for an
    unreadable video container.
    """

    return {
        "container_opened":
            False,

        "first_frame_readable":
            False,

        "container_fps":
            np.nan,

        "container_reported_frame_count":
            np.nan,

        "width":
            np.nan,

        "height":
            np.nan,

        "estimated_container_duration_seconds":
            np.nan,
    }


def probe_video(
    video_path,
):
    """
    Reads technical metadata from one video container.

    The reported FPS and duration describe the exported
    video container, not necessarily the original clinical
    capture timeline.

    The source file is not modified.
    """

    video_path = Path(
        video_path
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    try:
        if not cap.isOpened():
            return empty_video_probe()

        raw_fps = cap.get(
            cv2.CAP_PROP_FPS
        )

        raw_frame_count = cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )

        raw_width = cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )

        raw_height = cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )

        first_frame_readable, _ = (
            cap.read()
        )

    finally:
        cap.release()

    container_fps = (
        float(raw_fps)
        if (
            np.isfinite(raw_fps)
            and raw_fps > 0
        )
        else np.nan
    )


    container_reported_frame_count = (
        int(raw_frame_count)
        if (
            np.isfinite(raw_frame_count)
            and raw_frame_count > 0
            and float(raw_frame_count).is_integer()
        )
        else np.nan
    )


    width = (
        int(raw_width)
        if (
            np.isfinite(raw_width)
            and raw_width > 0
            and float(
                raw_width
            ).is_integer()
        )
        else np.nan
    )

    height = (
        int(raw_height)
        if (
            np.isfinite(raw_height)
            and raw_height > 0
            and float(
                raw_height
            ).is_integer()
        )
        else np.nan
    )

    estimated_container_duration_seconds = (
        float(container_reported_frame_count / container_fps)
        if (
            np.isfinite(container_reported_frame_count)
            and np.isfinite(container_fps)
        )
        else np.nan
    )


    return {
        "container_opened":
            True,

        "first_frame_readable":
            bool(
                first_frame_readable
            ),

        "container_fps":
            container_fps,

        "container_reported_frame_count":
            container_reported_frame_count,

        "width":
            width,

        "height":
            height,

        "estimated_container_duration_seconds":
            estimated_container_duration_seconds,
    }

In [13]:
# ------------------------------------------------------------------
# Build or reuse the video manifest after SHA-256 cache verification
# Replace the entire previous "Build video manifest" cell with this code.
# Requires: CONFIG, DIRS, df_clean, VIDEO_PROBE_COLUMNS, probe_video.
# Keep and run the preceding "Probe video metadata" definitions cell.
# ------------------------------------------------------------------


FORCE_REBUILD_MANIFEST = False
# Increment after changing probe_video() or the manifest construction rules.
MANIFEST_CACHE_VERSION = 1

VIDEO_MANIFEST_COLUMNS = [
    "video_key",
    "video_filename",
    "video_path",
    "video_relpath",
    "video_annotation_type",
    "video_size_bytes",
    *VIDEO_PROBE_COLUMNS,
]
MANIFEST_IDENTITY_COLUMNS = VIDEO_MANIFEST_COLUMNS[:6]
if len(VIDEO_MANIFEST_COLUMNS) != len(set(VIDEO_MANIFEST_COLUMNS)):
    raise ValueError("Duplicate column names in the manifest schema.")


def _manifest_walk_error(error):
    raise error


def _manifest_inventory(dataset_root, extensions, labelled_keys, storage_root):
    """Inventory physical files without opening video containers.

    The fingerprint uses path, category, size and modification time.
    Video contents are not hashed. video_relpath remains storage-root-relative.
    """
    records, seen_paths, paths_by_key = [], set(), {}
    for directory, subdirectories, filenames in os.walk(
        dataset_root, onerror=_manifest_walk_error, followlinks=False
    ):
        for name in subdirectories:
            if (Path(directory) / name).is_symlink():
                raise ValueError(f"Symbolic-link directory needs review: {name}")
        for filename in filenames:
            path = Path(directory) / filename
            if path.suffix.lower() not in extensions:
                continue
            path = path.resolve(strict=True)
            if not path.is_relative_to(dataset_root) or not path.is_file():
                raise ValueError(f"Invalid physical video path: {path}")
            key = path.stem
            if not key or path in seen_paths or key in paths_by_key:
                raise ValueError(
                    f"Duplicate/ambiguous video path or video_key: {path}; "
                    f"previous path for key {key!r}: {paths_by_key.get(key)}"
                )
            seen_paths.add(path)
            paths_by_key[key] = path
            stat = path.stat()
            records.append({
                "video_key": key,
                "video_filename": path.name,
                "video_path": str(path),
                "video_relpath": path.relative_to(storage_root).as_posix(),
                "video_annotation_type": (
                    "partially_labelled" if key in labelled_keys
                    else "fully_unlabelled"
                ),
                "video_size_bytes": stat.st_size,
                "video_mtime_ns": stat.st_mtime_ns,
            })
    return sorted(records, key=lambda record: record["video_key"])


def _manifest_sha256(path):
    # Hash the small manifest CSV, not the source videos.
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _manifest_atomic_write(path, value):
    handle, temporary_name = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    os.close(handle)
    temporary_path = Path(temporary_name)
    try:
        if isinstance(value, pd.DataFrame):
            value.to_csv(temporary_path, index=False)
        else:
            temporary_path.write_text(
                json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8"
            )
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


def _manifest_restore_types(table):
    """Use the same types for newly built and reloaded manifests."""
    if table.columns.tolist() != VIDEO_MANIFEST_COLUMNS:
        raise ValueError("The saved manifest has an unexpected column schema.")
    table = table.copy()
    for column in MANIFEST_IDENTITY_COLUMNS[:-1]:
        table[column] = table[column].astype("string")
    for column in ["container_opened", "first_frame_readable"]:
        values = table[column].astype("string")
        if not values.isin(["True", "False"]).all():
            raise ValueError(f"Invalid boolean values in {column}.")
        table[column] = values.map({"True": True, "False": False}).astype(bool)
    for column in ["video_size_bytes", "container_reported_frame_count", "width", "height"]:
        table[column] = table[column].replace("", pd.NA).astype("Int64")
    for column in ["container_fps", "estimated_container_duration_seconds"]:
        table[column] = pd.to_numeric(
            table[column].replace("", pd.NA), errors="raise"
        ).astype("float64")
    return table


def build_video_record(video_path, labelled_video_keys, storage_root):
    """Probe one physical video without modifying the source file."""
    video_path = Path(video_path).resolve(strict=True)
    storage_root = Path(storage_root).resolve()
    video_key = video_path.stem
    if not video_key:
        raise ValueError(f"Empty video_key: {video_path}")
    try:
        video_relpath = video_path.relative_to(storage_root).as_posix()
    except ValueError as error:
        raise ValueError(f"Video is outside storage_root: {video_path}") from error
    video_probe = probe_video(video_path)
    if set(video_probe) != set(VIDEO_PROBE_COLUMNS):
        raise ValueError("probe_video() output does not match VIDEO_PROBE_COLUMNS.")
    return {
        "video_key": video_key,
        "video_filename": video_path.name,
        "video_path": str(video_path),
        "video_relpath": video_relpath,
        "video_annotation_type": (
            "partially_labelled" if video_key in labelled_video_keys
            else "fully_unlabelled"
        ),
        "video_size_bytes": video_path.stat().st_size,
        **video_probe,
    }


def build_video_manifest(video_files, labelled_video_keys, storage_root):
    """Build one row per unique physical source video."""
    paths = tuple(Path(path).resolve(strict=True) for path in video_files)
    if not paths:
        raise ValueError("Cannot build a manifest without physical video files.")
    if len(set(paths)) != len(paths):
        raise ValueError("Duplicate physical video paths.")
    if len({path.stem for path in paths}) != len(paths):
        raise ValueError("Multiple physical videos share the same video_key.")
    records = [
        build_video_record(path, labelled_video_keys, storage_root)
        for path in tqdm(sorted(paths), desc="Building video manifest")
    ]
    return _manifest_restore_types(
        pd.DataFrame.from_records(records, columns=VIDEO_MANIFEST_COLUMNS)
        .sort_values("video_key", kind="stable")
        .reset_index(drop=True)
    )


manifest_root = Path(DIRS["dataset_root_dir"]).resolve()
manifest_storage_root = Path(CONFIG["storage_root"]).resolve()
manifest_dir = Path(DIRS["manifests_dir"])
manifest_path = manifest_dir / "video_manifest.csv"
manifest_state_path = manifest_dir / "video_manifest_state.json"
if not manifest_root.is_dir():
    raise FileNotFoundError(f"Dataset root is missing: {manifest_root}")
if not manifest_root.is_relative_to(manifest_storage_root):
    raise ValueError("dataset_root_dir must be inside the declared storage_root.")
if "video_key" not in df_clean.columns:
    raise ValueError("df_clean must contain video_key.")
manifest_dir.mkdir(parents=True, exist_ok=True)
manifest_extensions = {
    "." + str(ext).strip().lower().lstrip(".")
    for ext in CONFIG["dataset_validation"]["video_extensions"]
}
if not manifest_extensions or "." in manifest_extensions:
    raise ValueError("Configure at least one valid video extension.")
labelled_video_keys = frozenset(
    df_clean["video_key"].dropna().astype("string").str.strip()
    .loc[lambda values: values.ne("")].unique()
)
# Category membership is based on the current df_clean metadata.
print("Checking the current physical video inventory...")
manifest_inventory = _manifest_inventory(
    manifest_root, manifest_extensions, labelled_video_keys, manifest_storage_root
)
minimum_videos = int(CONFIG["dataset_validation"].get("minimum_video_files", 1))
if not manifest_inventory or len(manifest_inventory) < minimum_videos:
    raise ValueError(
        f"Found {len(manifest_inventory)} videos; configured minimum: {minimum_videos}."
    )
manifest_missing_keys = labelled_video_keys - {
    record["video_key"] for record in manifest_inventory
}
if manifest_missing_keys:
    raise ValueError(
        "Metadata videos missing from the physical inventory: "
        f"{sorted(manifest_missing_keys)[:10]}"
    )
# Refresh video_files for downstream cells; do not trust a previous runtime list.
video_files = [Path(record["video_path"]) for record in manifest_inventory]
manifest_expected_identity = [
    {column: record[column] for column in MANIFEST_IDENTITY_COLUMNS}
    for record in manifest_inventory
]
manifest_fingerprint_inputs = {
    "cache_version": MANIFEST_CACHE_VERSION,
    "columns": VIDEO_MANIFEST_COLUMNS,
    "opencv_version": cv2.__version__,
    "opencv_build_sha256": hashlib.sha256(cv2.getBuildInformation().encode()).hexdigest(),
    "dataset_root": str(manifest_root),
    "storage_root": str(manifest_storage_root),
    "video_extensions": sorted(manifest_extensions),
    "labelled_video_keys": sorted(labelled_video_keys),
    "inventory": manifest_inventory,
}
manifest_input_fingerprint = hashlib.sha256(
    json.dumps(manifest_fingerprint_inputs, sort_keys=True, ensure_ascii=False).encode()
).hexdigest()
video_manifest = None

if not FORCE_REBUILD_MANIFEST:
    try:
        manifest_state = json.loads(manifest_state_path.read_text(encoding="utf-8"))
        if manifest_state.get("complete") is not True:
            raise ValueError("The previous manifest build is incomplete.")
        if manifest_state.get("input_fingerprint") != manifest_input_fingerprint:
            raise ValueError("The source inventory, categories, schema or decoder changed.")
        if _manifest_sha256(manifest_path) != manifest_state.get("csv_sha256"):
            raise ValueError("The saved manifest CSV has changed.")
        candidate = _manifest_restore_types(pd.read_csv(
            manifest_path, dtype="string", keep_default_na=False
        ))
        if candidate[MANIFEST_IDENTITY_COLUMNS].to_dict("records") != manifest_expected_identity:
            raise ValueError("The saved manifest does not cover the current inventory exactly.")
        video_manifest = candidate
        print("Manifest unchanged: loaded saved CSV without calling probe_video().")
    except (OSError, ValueError, KeyError, TypeError, AttributeError) as error:
        print(f"Manifest rebuild required: {error}")
else:
    print("FORCE_REBUILD_MANIFEST=True: rebuilding the manifest.")

if video_manifest is None:
    # Prevent a stopped build from being mistaken for a complete new result.
    _manifest_atomic_write(manifest_state_path, {
        "complete": False,
        "input_fingerprint": manifest_input_fingerprint,
    })
    video_manifest = build_video_manifest(
        video_files, labelled_video_keys, manifest_storage_root
    )
    inventory_after = _manifest_inventory(
        manifest_root, manifest_extensions, labelled_video_keys, manifest_storage_root
    )
    if inventory_after != manifest_inventory:
        raise RuntimeError("Video files changed during probing. Stabilize the files and rerun.")
    if video_manifest[MANIFEST_IDENTITY_COLUMNS].to_dict("records") != manifest_expected_identity:
        raise ValueError("Built manifest does not match the source inventory.")
    _manifest_atomic_write(manifest_path, video_manifest)
    # Commit last. Complete means every video was probed, not every video passed.
    _manifest_atomic_write(manifest_state_path, {
        "complete": True,
        "cache_version": MANIFEST_CACHE_VERSION,
        "input_fingerprint": manifest_input_fingerprint,
        "csv_sha256": _manifest_sha256(manifest_path),
        "video_count": len(video_manifest),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    })
    print("Video manifest rebuilt and saved:", manifest_path)

print("Video manifest:", manifest_path)
print("Manifest cache state:", manifest_state_path)
print("Video manifest rows:", f"{len(video_manifest):,}")
display(video_manifest.head())
manifest_probe_issues = video_manifest.loc[
    ~video_manifest["container_opened"] | ~video_manifest["first_frame_readable"]
]
print("Videos with container/first-frame issues:", len(manifest_probe_issues))
if not manifest_probe_issues.empty:
    display(manifest_probe_issues)


Checking the current physical video inventory...
Manifest unchanged: loaded saved CSV without calling probe_video().
Video manifest: /content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests/video_manifest.csv
Manifest cache state: /content/drive/MyDrive/MMVQA_Clinical/data/curated/phase2/manifests/video_manifest_state.json
Video manifest rows: 117


,video_key,video_filename,video_path,video_relpath,video_annotation_type,video_size_bytes,container_opened,first_frame_readable,container_fps,container_reported_frame_count,width,height,estimated_container_duration_seconds
0,04a78ef00c5245e0,04a78ef00c5245e0.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,833894762,True,True,30.0,50986,336,336,1699.533333
1,0531325b64674948,0531325b64674948.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,263695527,True,True,30.0,16151,336,336,538.366667
2,055bbbec392b4f3a,055bbbec392b4f3a.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/unlabelled_...,fully_unlabelled,736094508,True,True,30.0,36533,336,336,1217.766667
3,0728084c8da942d9,0728084c8da942d9.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,798305289,True,True,30.0,48948,336,336,1631.600000
4,07c1fa15a20a4398,07c1fa15a20a4398.mp4,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,data/raw/kvasir_capsule/osfstorage/labelled_vi...,partially_labelled,603279074,True,True,30.0,38620,336,336,1287.333333


Videos with container/first-frame issues: 0


### 10. Validate the video manifest data

In [14]:
# ------------------------------------------------------------------
# GPU video-frame count audit with persistent per-video checkpoints.
# ------------------------------------------------------------------


DECODE_SETTINGS = {
    "requested_backend": CONFIG["decode_audit_backend"],
    "gpu_id": CONFIG["decode_audit_gpu_id"],
    "allow_cpu_fallback": CONFIG["decode_audit_allow_cpu_fallback"],
    "cpu_threads": CONFIG["decode_audit_cpu_threads"],
    "stage_video_locally": CONFIG["decode_audit_stage_video_locally"],
    "local_work_dir": CONFIG["decode_audit_local_work_dir"],
    "force_redecode": CONFIG["decode_audit_force_redecode"],
    "retry_failed": CONFIG["decode_audit_retry_failed"],
    "retry_video_keys": CONFIG["decode_audit_retry_video_keys"],
    "published_reference_frames": CONFIG["decode_audit_published_reference_frames"],
    "show_progress": True,
}



def _decode_canonical_json(value):
    """Stable JSON for cache checks; timestamps belong only to provenance."""
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
        allow_nan=False,
    )


def _decode_digest(value):
    return hashlib.sha256(_decode_canonical_json(value).encode("utf-8")).hexdigest()


def _decode_integer(value, name, minimum=None, nullable=False):
    from decimal import Decimal, InvalidOperation

    if value is None or value is pd.NA or (
        isinstance(value, str) and not value.strip()
    ):
        if nullable:
            return None
        raise ValueError(f"{name} must be an integer.")
    if isinstance(value, bool) or type(value).__name__ == "bool_":
        raise ValueError(f"{name} must not be boolean.")
    try:
        number = Decimal(str(value))
    except (InvalidOperation, TypeError, ValueError) as error:
        raise ValueError(f"{name} must be an integer: {value!r}") from error
    if not number.is_finite() or number != number.to_integral_value():
        raise ValueError(f"{name} must be a finite integer: {value!r}")
    number = int(number)
    if not -(2 ** 63) <= number < 2 ** 63:
        raise ValueError(f"{name} does not fit a signed 64-bit integer.")
    if minimum is not None and number < minimum:
        raise ValueError(f"{name} must be at least {minimum}.")
    return number


def _decode_source_identity(source):
    """Use original persistent source paths, never temporary staged paths."""
    identity = {}
    for column in ("video_key", "video_path", "video_relpath"):
        value = source.get(column)
        if value is None or value is pd.NA or not str(value).strip():
            raise ValueError(f"Source is missing {column}.")
        identity[column] = str(value)
    identity["video_size_bytes"] = _decode_integer(
        source.get("video_size_bytes"), "video_size_bytes", minimum=0
    )
    identity["video_mtime_ns"] = _decode_integer(
        source.get("video_mtime_ns"), "video_mtime_ns"
    )
    return identity


def decode_source_signature(source, decoder_signature, audit_version):
    """Hash source identity and decoder method, excluding category and run ID.

    The caller must supply a fresh inventory stat and verify it again after
    decoding. This signature hashes metadata; it does not hash video bytes.
    """
    if not isinstance(decoder_signature, dict) or not decoder_signature:
        raise ValueError("decoder_signature must describe the actual decoder.")
    return _decode_digest({
        "checkpoint_version": 1,
        "audit_version": _decode_integer(audit_version, "audit_version", minimum=1),
        "source_identity": _decode_source_identity(source),
        "decoder_signature": decoder_signature,
    })


def validate_decode_record(record, source):
    """Validate one completed attempt, which may explicitly report errors."""
    if not isinstance(record, dict):
        raise ValueError("A decode checkpoint must contain a record object.")
    source_identity = _decode_source_identity(source)
    if _decode_source_identity(record) != source_identity:
        raise ValueError("Decode record does not match the current video identity.")
    category = source.get("video_annotation_type")
    if category not in {"partially_labelled", "fully_unlabelled"}:
        raise ValueError("Source video category is invalid.")
    if record.get("video_annotation_type") != category:
        raise ValueError("Decode record category does not match its inventory.")
    statuses = {
        "read_completed", "open_failed", "no_readable_frames", "empty_frame",
        "decode_error",
    }
    stop_reason = record.get("stop_reason")
    if stop_reason not in statuses:
        raise ValueError(f"Unknown or unfinished decode status: {stop_reason!r}")
    frames_read = _decode_integer(
        record.get("frames_read_before_stop"), "frames_read_before_stop", minimum=0
    )
    reported = _decode_integer(
        record.get("container_reported_frames"), "container_reported_frames",
        minimum=0, nullable=True,
    )
    difference = _decode_integer(
        record.get("read_minus_reported"), "read_minus_reported", nullable=True
    )
    expected_difference = None if reported is None else frames_read - reported
    if difference != expected_difference:
        raise ValueError("read_minus_reported disagrees with the saved counts.")
    if stop_reason in {"open_failed", "no_readable_frames"} and frames_read:
        raise ValueError(f"{stop_reason} cannot have successfully decoded frames.")
    error_message = record.get("error_message")
    if error_message is None or error_message is pd.NA:
        error_message = ""
    if not isinstance(error_message, str):
        raise ValueError("error_message must be text or null.")
    decoding_completed = record.get("decoding_completed")
    if not isinstance(decoding_completed, bool):
        raise ValueError("decoding_completed must be a boolean.")
    if decoding_completed != (stop_reason in {"read_completed", "no_readable_frames"}):
        raise ValueError("decoding_completed and stop_reason disagree.")
    if stop_reason == "read_completed" and frames_read == 0:
        raise ValueError("A zero-frame completed read must be marked no_readable_frames.")
    backend = record.get("decoder_backend")
    if backend not in {"nvdec", "pyav_cpu"}:
        raise ValueError(f"Unknown decoder_backend: {backend!r}")
    fallback_reason = record.get("fallback_reason")
    if fallback_reason is None or fallback_reason is pd.NA:
        fallback_reason = ""
    if not isinstance(fallback_reason, str):
        raise ValueError("fallback_reason must be text.")
    decode_seconds = record.get("decode_seconds")
    if isinstance(decode_seconds, bool) or type(decode_seconds).__name__ == "bool_":
        raise ValueError("decode_seconds must be a finite nonnegative number.")
    try:
        decode_seconds = float(decode_seconds)
    except (TypeError, ValueError) as error:
        raise ValueError("decode_seconds must be a finite nonnegative number.") from error
    if not math.isfinite(decode_seconds) or decode_seconds < 0:
        raise ValueError("decode_seconds must be a finite nonnegative number.")
    # Preserve backend-specific JSON provenance while normalizing required fields.
    return {
        **record,
        "video_key": source_identity["video_key"],
        "video_annotation_type": category,
        "video_path": source_identity["video_path"],
        "video_relpath": source_identity["video_relpath"],
        "video_size_bytes": source_identity["video_size_bytes"],
        "video_mtime_ns": source_identity["video_mtime_ns"],
        "container_reported_frames": reported,
        "frames_read_before_stop": frames_read,
        "read_minus_reported": difference,
        "stop_reason": stop_reason,
        "error_message": error_message,
        "decoder_backend": backend,
        "fallback_reason": fallback_reason,
        "decode_seconds": decode_seconds,
        "decoding_completed": decoding_completed,
    }


def decode_record_needs_retry(record):
    """Retry only by explicit policy; terminal errors remain useful audit data."""
    return bool(
        not record["decoding_completed"]
        or record["frames_read_before_stop"] <= 0
        or bool(record.get("error_message"))
    )


def _decode_atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    temporary = Path(temporary_name)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8") as stream:
            stream.write(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def write_decode_checkpoint(path, source, signature, record, decoder_signature, audit_version):
    """Commit a terminal attempt atomically, including explicitly failed attempts.

    Do not call for KeyboardInterrupt or a partial, pending attempt. This
    checkpoint is independent of any global audit complete/quality flags.
    """
    from datetime import datetime, timezone

    expected = decode_source_signature(source, decoder_signature, audit_version)
    if signature != expected:
        raise ValueError("Checkpoint signature does not match the supplied source/decoder.")
    normalized = validate_decode_record(record, source)
    payload = {
        "checkpoint_version": 1,
        "complete": True,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "audit_version": _decode_integer(audit_version, "audit_version", minimum=1),
        "input_fingerprint": signature,
        "source_identity": _decode_source_identity(source),
        "decoder_signature": decoder_signature,
        "record": normalized,
        "record_sha256": _decode_digest(normalized),
    }
    payload["payload_sha256"] = _decode_digest(payload)
    _decode_atomic_json(path, payload)
    return payload


def read_decode_checkpoint(path, source, signature):
    """Read a verified terminal attempt and refresh its metadata category.

    A previously failed attempt is intentionally reusable: the caller may
    choose to retry it with decode_record_needs_retry and retry_failed=True.
    No global audit state or category membership enters this cache decision.
    """
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError("Invalid decode checkpoint object.")
    if payload.get("checkpoint_version") != 1 or payload.get("complete") is not True:
        raise ValueError("Decode checkpoint schema is unknown or attempt incomplete.")
    content = {key: value for key, value in payload.items() if key != "payload_sha256"}
    if payload.get("payload_sha256") != _decode_digest(content):
        raise ValueError("Decode checkpoint payload changed.")
    if payload.get("input_fingerprint") != signature:
        raise ValueError("Decode checkpoint source or decoder changed.")
    if payload.get("source_identity") != _decode_source_identity(source):
        raise ValueError("Decode checkpoint does not match the current inventory.")
    expected = decode_source_signature(source, payload["decoder_signature"], payload["audit_version"])
    if expected != signature:
        raise ValueError("Checkpoint provenance disagrees with its fingerprint.")
    record = payload.get("record")
    if not isinstance(record, dict) or payload.get("record_sha256") != _decode_digest(record):
        raise ValueError("Decode checkpoint record changed.")
    recorded_source = {**source, "video_annotation_type": record.get("video_annotation_type")}
    normalized = validate_decode_record(record, recorded_source)
    # Decoding pixels does not depend on which current metadata rows label a video.
    normalized["video_annotation_type"] = source["video_annotation_type"]
    normalized = validate_decode_record(normalized, source)
    return normalized, payload


def decode_json_digest(value):
    return _decode_digest(value)


def decode_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def decode_atomic_write(path, value):
    path = Path(path)
    if not isinstance(value, pd.DataFrame):
        _decode_atomic_json(path, value)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    os.close(descriptor)
    temporary = Path(temporary_name)
    try:
        value.to_csv(temporary, index=False)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


"""Full-file NVDEC/PyAV frame counting. Dependencies are imported on demand."""


class DecodeBackendAPIError(RuntimeError):
    """An incompatible decoder binding or programming error, not bad video data."""


def _require_decode_api(owner, *method_names):
    """Reject incompatible bindings before compressed packets are decoded."""
    missing = [name for name in method_names if not callable(getattr(owner, name, None))]
    if missing:
        raise DecodeBackendAPIError(
            f"{type(owner).__name__} is missing required callable API: {', '.join(missing)}. "
            "Update the decoder code/package; CPU fallback is not used for API mismatches."
        )


def ensure_gpu_available(gpu_id=0):
    """Check accessible CUDA hardware before starting an NVDEC audit.

    This checks CUDA/driver libraries, not support for a particular video codec.
    No GPU or missing driver libraries is a startup error, not a CPU fallback.
    """
    import ctypes
    import ctypes.util
    import operator

    if isinstance(gpu_id, bool):
        raise ValueError("gpu_id must be a nonnegative integer.")
    gpu_id = operator.index(gpu_id)
    if gpu_id < 0:
        raise ValueError("gpu_id must be a nonnegative integer.")
    try:
        cuda = ctypes.CDLL(ctypes.util.find_library("cuda") or "libcuda.so.1")
        # CUDA compute alone does not guarantee the video driver was mounted.
        ctypes.CDLL(ctypes.util.find_library("nvcuvid") or "libnvcuvid.so.1")
    except OSError as error:
        raise RuntimeError(
            "NVDEC requires an NVIDIA GPU runtime with libcuda and libnvcuvid. "
            "Select a GPU runtime and verify the video driver is accessible. "
            f"Driver loading error: {error}"
        ) from error

    signatures = {
        "cuInit": [ctypes.c_uint],
        "cuDeviceGetCount": [ctypes.POINTER(ctypes.c_int)],
        "cuDeviceGet": [ctypes.POINTER(ctypes.c_int), ctypes.c_int],
        "cuDeviceGetName": [ctypes.POINTER(ctypes.c_char), ctypes.c_int, ctypes.c_int],
        "cuDriverGetVersion": [ctypes.POINTER(ctypes.c_int)],
    }
    for name, arguments in signatures.items():
        function = getattr(cuda, name)
        function.argtypes = arguments
        function.restype = ctypes.c_int

    def check(name, *arguments):
        status = getattr(cuda, name)(*arguments)
        if status != 0:
            raise RuntimeError(
                f"NVIDIA GPU preflight failed: {name} returned CUDA status {status}."
            )

    check("cuInit", 0)
    count = ctypes.c_int()
    check("cuDeviceGetCount", ctypes.byref(count))
    if gpu_id >= count.value:
        raise RuntimeError(
            f"Requested GPU {gpu_id}, but CUDA exposes {count.value} device(s)."
        )
    device = ctypes.c_int()
    check("cuDeviceGet", ctypes.byref(device), gpu_id)
    name = ctypes.create_string_buffer(256)
    check("cuDeviceGetName", name, len(name), device.value)
    driver = ctypes.c_int()
    check("cuDriverGetVersion", ctypes.byref(driver))
    return {
        "gpu_id": gpu_id,
        "gpu_name": name.value.decode("utf-8", errors="replace"),
        "cuda_visible_device_count": count.value,
        "cuda_driver_api_version": driver.value,
        "nvdec_driver_library_available": True,
    }


def _decode_positive_header_count(value):
    """Unknown/zero container counts remain missing, never fabricated."""
    try:
        count = int(value)
        return count if count > 0 and count == value else None
    except (TypeError, ValueError, OverflowError):
        return None


def _decode_header_probe(source_path):
    """Read header metadata only; this never calls decode or reads frame pixels."""
    result = {
        "container_reported_frames": None,
        "header_source": "unavailable",
        "header_probe_error": None,
    }
    try:
        import av
        with av.open(str(source_path), mode="r") as container:
            streams = list(container.streams.video)
            if len(streams) != 1:
                result["header_probe_error"] = (
                    f"Expected one video stream for an unambiguous header count; found {len(streams)}."
                )
            else:
                result["container_reported_frames"] = _decode_positive_header_count(streams[0].frames)
                result["header_source"] = "pyav_container_header"
    except Exception as error:
        result["header_probe_error"] = f"{type(error).__name__}: {error}"
    return result


def _decode_result(backend):
    return {
        "container_reported_frames": None,
        "frames_read_before_stop": 0,
        "read_minus_reported": None,
        "stop_reason": "open_failed",
        "error_message": None,
        "decoder_backend": backend,
        "fallback_reason": None,
        "decode_seconds": 0.0,
        "decoding_completed": False,
        "header_source": "unavailable",
        "header_probe_error": None,
        "decoded_width": None,
        "decoded_height": None,
        "decoder_package_version": None,
        "packets_processed": None,
        "frames_returned_during_flush": None,
    }


def _decode_progress(source_path, settings, backend):
    if not settings.get("show_progress", True):
        return None
    from pathlib import Path
    from tqdm.auto import tqdm
    # Unknown total: header counts are diagnostics, not decoding bounds.
    return tqdm(total=None, desc=f"{backend}: {Path(source_path).name}",
                unit="frame", mininterval=1.0, leave=False)


def _decode_finish(result, started):
    from time import perf_counter
    result["decode_seconds"] = perf_counter() - started
    reported = result["container_reported_frames"]
    result["read_minus_reported"] = (
        result["frames_read_before_stop"] - reported if reported is not None else None
    )
    return result


def _decode_with_nvdec(source_path, settings):
    """Decode all packets, including the demuxer's final empty EOS packet.

    PyNvVideoCodec 2.2.3 yields one terminal PacketData with bsl == 0.
    Decode(EOS) drains delayed frames. Its PyNvDecoder has no Flush method.
    """
    from time import perf_counter
    from importlib.metadata import version
    started = perf_counter()
    result = _decode_result("nvdec")
    result.update(_decode_header_probe(source_path))
    decoder = demuxer = progress = None
    stage = "open"
    try:
        import PyNvVideoCodec as nvc
        result["decoder_package_version"] = version("PyNvVideoCodec")
        _require_decode_api(nvc, "CreateDemuxer", "CreateDecoder")
        demuxer = nvc.CreateDemuxer(filename=str(source_path))
        _require_decode_api(demuxer, "GetNvCodecId", "__iter__")
        decoder = nvc.CreateDecoder(
            gpuid=int(settings.get("gpu_id", 0)),
            codec=demuxer.GetNvCodecId(),
            usedevicememory=True,
        )
        _require_decode_api(decoder, "Decode", "GetWidth", "GetHeight", "SyncOnCUStream")
        result["packets_processed"] = 0
        result["frames_returned_during_flush"] = 0
        progress = _decode_progress(source_path, settings, "NVDEC")
        stage = "decode"

        def count_frames(frames, is_flush=False):
            previous_count = result["frames_read_before_stop"]
            try:
                for frame in frames:
                    if frame is None:
                        raise RuntimeError("NVDEC returned an empty decoded frame.")
                    if result["frames_read_before_stop"] == 0:
                        _require_decode_api(frame, "framesize")
                    if frame.framesize() <= 0:
                        raise RuntimeError("NVDEC returned an empty decoded frame.")
                    width, height = decoder.GetWidth(), decoder.GetHeight()
                    if width <= 0 or height <= 0:
                        raise RuntimeError("NVDEC returned invalid frame dimensions.")
                    result["decoded_width"], result["decoded_height"] = int(width), int(height)
                    result["frames_read_before_stop"] += 1
                    if is_flush:
                        result["frames_returned_during_flush"] += 1
            finally:
                if progress is not None:
                    progress.update(result["frames_read_before_stop"] - previous_count)

        eos_seen = False
        for packet in demuxer:
            # The real 2.2.3 demux iterator supplies EOS itself. Do not send a
            # second EOS or call an undocumented Flush/EndDecode method.
            is_eos = packet.bsl == 0
            stage = "drain" if is_eos else "decode"
            count_frames(decoder.Decode(packet), is_flush=is_eos)
            result["packets_processed"] += 1
            if is_eos:
                eos_seen = True
                break
        if not eos_seen:
            raise DecodeBackendAPIError(
                "The PyNvVideoCodec demux iterator ended without its terminal bsl=0 EOS packet. "
                "Decoder draining cannot be confirmed; CPU fallback is not used for this API mismatch."
            )
        stage = "synchronize"
        decoder.SyncOnCUStream()
        result["decoding_completed"] = True
        result["stop_reason"] = (
            "read_completed" if result["frames_read_before_stop"] > 0 else "no_readable_frames"
        )
    except DecodeBackendAPIError:
        raise
    except (AttributeError, TypeError) as error:
        raise DecodeBackendAPIError(
            f"NVDEC API mismatch during {stage}: {error}. "
            "CPU fallback is not used for incompatible API attributes or signatures."
        ) from error
    except Exception as error:
        result["stop_reason"] = "open_failed" if stage == "open" else "decode_error"
        result["error_message"] = f"{stage}: {type(error).__name__}: {error}"
    finally:
        if progress is not None:
            progress.close()
        # Native resources are released as these local owners leave scope.
        decoder = None
        demuxer = None
    return _decode_finish(result, started)


def _decode_with_pyav(source_path, settings):
    """Explicit software fallback; count actual frames, including decoder flush."""
    from time import perf_counter
    started = perf_counter()
    result = _decode_result("pyav_cpu")
    progress = None
    stage = "open"
    try:
        import av
        result["decoder_package_version"] = av.__version__
        with av.open(str(source_path), mode="r") as container:
            if not container.streams.video:
                raise ValueError("No video stream exists in this container.")
            stream = container.streams.video[0]
            if len(container.streams.video) == 1:
                result["container_reported_frames"] = _decode_positive_header_count(stream.frames)
                result["header_source"] = "pyav_container_header"
            else:
                result["header_probe_error"] = "Multiple video streams; auditing the first video stream."
            stream.codec_context.thread_count = int(settings.get("cpu_threads", 2))
            progress = _decode_progress(source_path, settings, "PyAV CPU")
            stage = "decode"
            # PyAV demux yields terminal flush packets; container.decode consumes
            # those too. No loop bound is inferred from stream.frames.
            for frame in container.decode(stream):
                if frame is None or frame.width <= 0 or frame.height <= 0:
                    raise RuntimeError("PyAV returned an empty decoded frame.")
                result["decoded_width"], result["decoded_height"] = int(frame.width), int(frame.height)
                result["frames_read_before_stop"] += 1
                if progress is not None:
                    progress.update(1)
        result["decoding_completed"] = True
        result["stop_reason"] = (
            "read_completed" if result["frames_read_before_stop"] > 0 else "no_readable_frames"
        )
    except Exception as error:
        result["stop_reason"] = "open_failed" if stage == "open" else "decode_error"
        result["error_message"] = f"{stage}: {type(error).__name__}: {error}"
    finally:
        if progress is not None:
            progress.close()
    return _decode_finish(result, started)


def decode_video_gpu(source_path, settings):
    """Audit one file; an allowed CPU fallback restarts the entire file.

    Call ensure_gpu_available once before uncached GPU work, in the driver.
    KeyboardInterrupt/SystemExit deliberately propagate and must not be cached.
    """
    from pathlib import Path
    from time import perf_counter
    import operator

    source_path = Path(source_path)
    requested = settings.get("requested_backend", "nvdec")
    if requested not in {"nvdec", "pyav_cpu"}:
        raise ValueError("requested_backend must be 'nvdec' or 'pyav_cpu'.")
    for key, default, minimum in (("gpu_id", 0, 0), ("cpu_threads", 2, 1)):
        value = settings.get(key, default)
        if isinstance(value, bool) or operator.index(value) < minimum:
            raise ValueError(f"{key} must be an integer >= {minimum}.")
    for key, default in (("allow_cpu_fallback", True), ("show_progress", True)):
        if not isinstance(settings.get(key, default), bool):
            raise ValueError(f"{key} must be boolean.")

    if requested == "pyav_cpu":
        return _decode_with_pyav(source_path, settings)
    started = perf_counter()
    result = _decode_with_nvdec(source_path, settings)
    if (result["decoding_completed"] and result["frames_read_before_stop"] > 0) or not settings.get("allow_cpu_fallback", True):
        return result

    # Never mix the partial GPU count with the complete CPU attempt.
    cpu_result = _decode_with_pyav(source_path, settings)
    cpu_result["fallback_reason"] = (
        f"NVDEC {result['stop_reason']}: {result['error_message'] or 'No frames returned.'}"
    )
    cpu_result["gpu_attempt_frames_read"] = result["frames_read_before_stop"]
    cpu_result["gpu_attempt_seconds"] = result["decode_seconds"]
    cpu_result["cpu_attempt_seconds"] = cpu_result["decode_seconds"]
    cpu_result["decode_seconds"] = perf_counter() - started
    return cpu_result


# ------------------------------------------------------------------
# Inventory and orchestration for a persistent per-video decode audit
# ------------------------------------------------------------------
DECODE_AUDIT_VERSION = 2
DECODE_IDENTITY_COLUMNS = [
    "video_key", "video_annotation_type", "video_path", "video_relpath",
    "video_size_bytes", "video_mtime_ns",
]
DECODE_COUNT_COLUMNS = [
    "container_reported_frames", "frames_read_before_stop", "read_minus_reported",
]


def scan_decode_inventory(root, extensions, labelled_keys):
    """Fresh file metadata inventory; video contents are not hashed here."""
    root = Path(root).resolve(strict=True)
    records, seen_keys, seen_paths = [], set(), set()
    def fail(error):
        raise error
    for directory, subdirectories, filenames in os.walk(root, onerror=fail, followlinks=False):
        for name in subdirectories:
            if (Path(directory) / name).is_symlink():
                raise ValueError(f"Review symbolic-link directory: {Path(directory) / name}")
        for name in filenames:
            path = Path(directory) / name
            if path.suffix.lower() not in extensions:
                continue
            physical = path.resolve(strict=True)
            if not physical.is_relative_to(root) or not physical.is_file():
                raise ValueError(f"Invalid video path: {path}")
            key = path.stem
            if not key or key in seen_keys or physical in seen_paths:
                raise ValueError(f"Duplicate video identifier or physical path: {path}")
            seen_keys.add(key)
            seen_paths.add(physical)
            info = physical.stat()
            records.append({
                "video_key": key,
                "video_annotation_type": "partially_labelled" if key in labelled_keys else "fully_unlabelled",
                "video_path": str(physical),
                "video_relpath": physical.relative_to(root).as_posix(),
                "video_size_bytes": info.st_size, "video_mtime_ns": info.st_mtime_ns,
            })
    return sorted(records, key=lambda row: row["video_key"])


def _decode_check_source_unchanged(source):
    info = Path(source["video_path"]).stat()
    if (info.st_size, info.st_mtime_ns) != (source["video_size_bytes"], source["video_mtime_ns"]):
        raise RuntimeError(f"Source changed during audit: {source['video_path']}")


def _decode_installed_version(name):
    try:
        return package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        return None


def make_decode_signature(settings):
    """Version the counting method, independently of run timestamps and labels."""
    signature = {
        "method": "demux_decode_eos_native_frames_v3",
        "requested_backend": settings["requested_backend"],
        "pyav_version": _decode_installed_version("av"),
        "allow_cpu_fallback": settings["allow_cpu_fallback"],
    }
    if settings["requested_backend"] == "nvdec":
        signature.update(pynvvideocodec_version=_decode_installed_version("PyNvVideoCodec"),
                         gpu_id=settings["gpu_id"])
    return signature


@contextmanager
def local_decode_source(source, settings):
    """Copy only the current compressed video; preserve original paths in cache."""
    path = Path(source["video_path"])
    if not settings["stage_video_locally"]:
        yield path, 0.0
        return
    local_root = Path(settings["local_work_dir"])
    local_root.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(local_root).free < source["video_size_bytes"] + 256 * 1024**2:
        raise OSError("Insufficient local space for current video. Free space or disable local staging.")
    with tempfile.TemporaryDirectory(prefix="decode-", dir=local_root) as folder:
        target = Path(folder) / path.name
        tqdm.write(f"Copying to local disk: {path.name}")
        tick = time.perf_counter()
        shutil.copy2(path, target)
        copy_seconds = time.perf_counter() - tick
        _decode_check_source_unchanged(source)
        yield target, copy_seconds


def _decode_existing_report_matches(state_path, output_paths, report_fingerprint):
    try:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        return bool(
            state.get("complete") is True
            and state.get("audit_version") == DECODE_AUDIT_VERSION
            and state.get("report_fingerprint") == report_fingerprint
            and set(state.get("output_sha256", {})) == {p.name for p in output_paths}
            and all(state["output_sha256"][p.name] == decode_file_sha256(p) for p in output_paths)
        )
    except (OSError, ValueError, TypeError, KeyError, AttributeError):
        return False


def _decode_preserve_legacy_outputs(results_dir, state_path, paths):
    """Keep the prior OpenCV reports when publishing the first new audit."""
    try:
        previous = json.loads(state_path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        previous = {}
    if isinstance(previous, dict) and previous.get("audit_version") == DECODE_AUDIT_VERSION:
        return
    existing = [path for path in [*paths, state_path, results_dir / "decode_audit.partial.csv"] if path.is_file()]
    if not existing:
        return
    backup = results_dir / "decode_audit_legacy_backup" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    backup.mkdir(parents=True, exist_ok=False)
    for path in existing:
        shutil.copy2(path, backup / path.name)
    print("Previous reports preserved:", backup)


def run_gpu_decode_audit(dirs, config, metadata, settings):
    """Decode new/changed videos; reuse completed attempts including flagged ones."""
    settings = dict(settings)
    root = Path(dirs["dataset_root_dir"]).resolve(strict=True)
    if not root.is_dir():
        raise ValueError(f"Dataset root is not a directory: {root}")
    if "video_key" not in metadata:
        raise ValueError("df_clean needs video_key.")
    if settings["requested_backend"] not in {"nvdec", "pyav_cpu"}:
        raise ValueError("requested_backend must be nvdec or pyav_cpu.")
    for key in ("gpu_id", "cpu_threads"):
        if isinstance(settings[key], bool) or not isinstance(settings[key], int):
            raise ValueError(f"{key} must be an integer.")
    if settings["gpu_id"] < 0 or settings["cpu_threads"] < 1:
        raise ValueError("gpu_id must be nonnegative and cpu_threads positive.")
    for key in ("allow_cpu_fallback", "force_redecode", "retry_failed", "stage_video_locally", "show_progress"):
        if not isinstance(settings[key], bool):
            raise ValueError(f"{key} must be boolean.")
    if not isinstance(settings["retry_video_keys"], (list, tuple, set, frozenset)):
        raise ValueError("retry_video_keys must be a collection of video keys, not a string.")
    retry_keys = {str(key) for key in settings["retry_video_keys"]}
    settings["retry_video_keys"] = sorted(retry_keys)
    settings["local_work_dir"] = str(settings["local_work_dir"])
    labelled_keys = frozenset(
        metadata["video_key"].dropna().astype("string").str.strip().loc[lambda value: value.ne("")]
    )
    extensions = {"." + str(ext).strip().lower().lstrip(".")
                  for ext in config["dataset_validation"]["video_extensions"]}
    if not extensions or "." in extensions:
        raise ValueError("Configure valid video extensions.")
    print("Checking the current physical video inventory...")
    inventory = scan_decode_inventory(root, extensions, labelled_keys)
    minimum = int(config["dataset_validation"].get("minimum_video_files", 1))
    if not inventory or len(inventory) < minimum:
        raise ValueError(f"Found {len(inventory)} videos; configured minimum: {minimum}.")
    present_keys = {row["video_key"] for row in inventory}
    missing_keys = labelled_keys - present_keys
    if missing_keys:
        raise ValueError(f"Metadata videos missing physically: {sorted(missing_keys)[:10]}")
    if retry_keys - present_keys:
        raise ValueError(f"Unknown retry_video_keys: {sorted(retry_keys - present_keys)}")
    categories = ("partially_labelled", "fully_unlabelled")
    if {row["video_annotation_type"] for row in inventory} != set(categories):
        raise ValueError("This dataset audit expects both labelled and unlabelled video groups.")
    references = {key: int(value) for key, value in settings["published_reference_frames"].items()}
    if set(references) != {*categories, "all_videos"} or any(value < 0 for value in references.values()):
        raise ValueError("Provide nonnegative published references for both groups and all_videos.")
    if references["all_videos"] != sum(references[key] for key in categories):
        raise ValueError("Published group references must sum to the total reference.")

    results_dir = Path(dirs["results_dir"])
    results_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = results_dir / "decode_audit_cache_v2"
    cache_dir.mkdir(exist_ok=True)
    state_path = results_dir / "decode_audit_state.json"
    run_state_path = results_dir / "decode_audit_run_state.json"
    signature_info = make_decode_signature(settings)
    input_fingerprint = decode_json_digest({"inventory": inventory, "decoder": signature_info,
                                            "audit_version": DECODE_AUDIT_VERSION})
    try:
        old_state = json.loads(state_path.read_text(encoding="utf-8"))
        if old_state.get("complete") is not True:
            print("Previous global audit was marked incomplete; checking individual video checkpoints.")
        if old_state.get("audit_version") != DECODE_AUDIT_VERSION:
            print("Previous OpenCV results are not NVDEC checkpoints; this first upgrade decodes once.")
    except (OSError, ValueError, AttributeError):
        pass
    run_state = {
        "complete": False, "started_utc": datetime.now(timezone.utc).isoformat(),
        "input_fingerprint": input_fingerprint, "settings": settings,
        "videos_total": len(inventory), "videos_completed": 0,
    }
    decode_atomic_write(run_state_path, run_state)
    records = []
    reused = rebuilt = 0
    hardware = None
    for source in tqdm(inventory, desc="Auditing videos", unit="video", disable=not settings["show_progress"]):
        key = source["video_key"]
        signature = decode_source_signature(source, signature_info, DECODE_AUDIT_VERSION)
        checkpoint_path = cache_dir / (hashlib.sha256(key.encode()).hexdigest() + ".json")
        record = None
        if not settings["force_redecode"] and key not in retry_keys:
            try:
                record, _ = read_decode_checkpoint(checkpoint_path, source, signature)
                if settings["retry_failed"] and decode_record_needs_retry(record):
                    record = None
            except (OSError, ValueError, KeyError, TypeError, AttributeError):
                record = None
        if record is None:
            # GPU validation is lazy: a fully cached run needs no active GPU.
            if settings["requested_backend"] == "nvdec" and hardware is None:
                hardware = ensure_gpu_available(settings["gpu_id"])
                # Installation/runtime problems must stop here, not turn every
                # video into a silently cached CPU fallback.
                import PyNvVideoCodec as _nvc
                import av as _av
                print("GPU decode device:", hardware)
            elif settings["requested_backend"] == "pyav_cpu":
                import av as _av
            with local_decode_source(source, settings) as (path, copy_seconds):
                result = decode_video_gpu(path, settings)
            _decode_check_source_unchanged(source)
            record = {**source, **result, "copy_seconds": copy_seconds}
            record["gpu_device"] = hardware if settings["requested_backend"] == "nvdec" else None
            record = validate_decode_record(record, source)
            write_decode_checkpoint(checkpoint_path, source, signature, record,
                                    signature_info, DECODE_AUDIT_VERSION)
            rebuilt += 1
            if record.get("fallback_reason"):
                tqdm.write(f"CPU fallback for {key}: {record['fallback_reason']}")
        else:
            reused += 1
        records.append(record)
        run_state.update(videos_completed=len(records), videos_reused=reused, videos_decoded=rebuilt)
        decode_atomic_write(run_state_path, run_state)
        if decode_record_needs_retry(record):
            tqdm.write(f"Review {key}: {record['stop_reason']} — {record['error_message']}")
    if scan_decode_inventory(root, extensions, labelled_keys) != inventory:
        raise RuntimeError("Sources changed during this run; completed video checkpoints remain saved.")

    table = pd.DataFrame.from_records(records)
    for column in ["video_size_bytes", "video_mtime_ns", *DECODE_COUNT_COLUMNS]:
        table[column] = pd.array(table[column], dtype="Int64")
    # Flatten nested provenance in CSV, keeping the structured form in checkpoints.
    for column in table.columns:
        if table[column].map(lambda value: isinstance(value, (dict, list))).any():
            table[column] = table[column].map(
                lambda value: json.dumps(value, sort_keys=True) if isinstance(value, (dict, list)) else value
            )
    table = table.sort_values("video_key", kind="stable").reset_index(drop=True)
    table["needs_review"] = (
        ~table["decoding_completed"].astype(bool)
        | table["frames_read_before_stop"].le(0)
        | table["read_minus_reported"].isna()
        | table["read_minus_reported"].ne(0)
        | table["error_message"].fillna("").ne("")
    ).fillna(True)
    summary = table.groupby("video_annotation_type", sort=True).agg(
        video_count=("video_key", "size"), frames_read_before_stop=("frames_read_before_stop", "sum"),
        fully_processed_videos=("decoding_completed", "sum"), videos_needing_review=("needs_review", "sum"),
    )
    summary.loc["all_videos"] = summary.sum()
    summary["published_reference_frames"] = summary.index.map(references)
    summary["read_minus_reference"] = summary["frames_read_before_stop"] - summary["published_reference_frames"]
    summary["all_videos_processed"] = summary["fully_processed_videos"].eq(summary["video_count"])
    summary = summary.rename_axis("video_annotation_type")
    output_tables = {
        results_dir / "decode_audit.csv": table,
        results_dir / "labelled_video_decode_audit.csv": table.loc[table["video_annotation_type"].eq(categories[0])],
        results_dir / "unlabelled_video_decode_audit.csv": table.loc[table["video_annotation_type"].eq(categories[1])],
        results_dir / "decode_audit_summary.csv": summary.reset_index(),
    }
    report_fingerprint = decode_json_digest({"input_fingerprint": input_fingerprint,
                                             "records": records, "published_reference_frames": references})
    if _decode_existing_report_matches(state_path, output_tables, report_fingerprint):
        print("Verified SHA-256: reusing unchanged published CSVs.")
    else:
        _decode_preserve_legacy_outputs(results_dir, state_path, output_tables)
        # Publication has its own complete flag; checkpoints survive interruption.
        decode_atomic_write(state_path, {"complete": False, "audit_version": DECODE_AUDIT_VERSION,
                                          "input_fingerprint": input_fingerprint})
        for path, value in output_tables.items():
            decode_atomic_write(path, value)
        decode_atomic_write(state_path, {
            "complete": True, "audit_version": DECODE_AUDIT_VERSION,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "input_fingerprint": input_fingerprint, "report_fingerprint": report_fingerprint,
            "video_count": len(table), "all_videos_decoded": bool(
                table["decoding_completed"].all() and table["frames_read_before_stop"].gt(0).all()
            ),
            "videos_needing_review": int(table["needs_review"].sum()),
            "decoder_signature": signature_info,
            "output_sha256": {path.name: decode_file_sha256(path) for path in output_tables},
            "source_fingerprint_method": "SHA-256 of paths, sizes, mtimes and decoder settings; not video bytes",
        })
    run_state.update(complete=True, finished_utc=datetime.now(timezone.utc).isoformat())
    decode_atomic_write(run_state_path, run_state)
    print(f"Videos decoded this run: {rebuilt}; checkpoints reused: {reused}.")
    print("Backends recorded in audit:", table["decoder_backend"].value_counts().to_dict())
    print("Audit:", results_dir / "decode_audit.csv")
    print("Cache state:", state_path)
    print("Decoder results are observations; completion does not certify absence of concealed codec errors.")
    display(summary)
    display(table.loc[table["needs_review"]])
    return table, summary


# Execute the configured audit and expose its two result tables.
decode_audit, decode_summary = run_gpu_decode_audit(
    dirs=DIRS, config=CONFIG, metadata=df_clean, settings=DECODE_SETTINGS,
)


Checking the current physical video inventory...


Auditing videos:   0%|          | 0/117 [00:00<?, ?video/s]

Verified SHA-256: reusing unchanged published CSVs.
Videos decoded this run: 0; checkpoints reused: 117.
Backends recorded in audit: {'nvdec': 117}
Audit: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results/decode_audit.csv
Cache state: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results/decode_audit_state.json
Decoder results are observations; completion does not certify absence of concealed codec errors.


,video_count,frames_read_before_stop,fully_processed_videos,videos_needing_review,published_reference_frames,read_minus_reference,all_videos_processed
video_annotation_type,,,,,,,
fully_unlabelled,74,2785829,74,0,2785829,0,True
partially_labelled,43,1979285,43,0,1955675,23610,True
all_videos,117,4765114,117,0,4741504,23610,True


,video_key,video_annotation_type,video_path,video_relpath,video_size_bytes,video_mtime_ns,container_reported_frames,frames_read_before_stop,read_minus_reported,stop_reason,...,header_source,header_probe_error,decoded_width,decoded_height,decoder_package_version,packets_processed,frames_returned_during_flush,copy_seconds,gpu_device,needs_review


In [15]:
def validate_video_manifest(
    dataframe,
    labelled_video_keys,
    config,
):
    """
    Validates the complete physical video inventory,
    technical probe results, and consistency with
    labelled metadata.

    The input DataFrame is not modified.
    """

    required_columns = set(
        VIDEO_MANIFEST_COLUMNS
    )

    missing_columns = sorted(
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            "Video manifest is missing required columns: "
            f"{missing_columns}"
        )

    if dataframe.empty:
        raise ValueError(
            "Video manifest cannot be empty."
        )

    # --------------------------------------------------------------
    # Validate normalized video identifiers
    # --------------------------------------------------------------

    video_keys = (
        dataframe[
            "video_key"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_key_mask = (
        video_keys.isna()
        | video_keys.eq("")
    )

    if invalid_video_key_mask.any():
        raise ValueError(
            "Video manifest contains "
            f"{int(invalid_video_key_mask.sum())} "
            "missing or empty video keys."
        )

    duplicate_video_keys = (
        video_keys.loc[
            video_keys.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_keys:
        raise ValueError(
            "Multiple physical videos resolve to the same "
            f"video key: {duplicate_video_keys}"
        )

    # --------------------------------------------------------------
    # Validate physical and relative paths
    # --------------------------------------------------------------

    video_paths = (
        dataframe[
            "video_path"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_video_path_mask = (
        video_paths.isna()
        | video_paths.eq("")
    )

    if invalid_video_path_mask.any():
        raise ValueError(
            "Video manifest contains "
            f"{int(invalid_video_path_mask.sum())} "
            "missing video paths."
        )

    physical_file_exists = (
        video_paths.map(
            lambda value:
                Path(value).is_file()
        )
    )

    if not physical_file_exists.all():
        missing_files = (
            video_paths.loc[
                ~physical_file_exists
            ]
            .tolist()
        )

        raise FileNotFoundError(
            "Video manifest contains paths that do not "
            f"exist: {missing_files}"
        )

    duplicate_video_paths = (
        video_paths.loc[
            video_paths.duplicated(
                keep=False
            )
        ]
        .unique()
        .tolist()
    )

    if duplicate_video_paths:
        raise ValueError(
            "Duplicate physical video paths found: "
            f"{duplicate_video_paths}"
        )

    relative_paths = (
        dataframe[
            "video_relpath"
        ]
        .astype("string")
        .str.strip()
    )

    invalid_relative_path_mask = (
        relative_paths.isna()
        | relative_paths.eq("")
        | relative_paths.map(
            lambda value:
                (
                    Path(value).is_absolute()
                    if pd.notna(value)
                    else True
                )
        )
    )

    if invalid_relative_path_mask.any():
        raise ValueError(
            "Video manifest contains invalid or absolute "
            "video-relative paths."
        )

    if relative_paths.duplicated().any():
        raise ValueError(
            "Video manifest contains duplicate "
            "video-relative paths."
        )

    # --------------------------------------------------------------
    # Validate complete physical inventory
    # --------------------------------------------------------------

    actual_total_videos = len(dataframe)

    minimum_video_files = int(
        config["dataset_validation"]["minimum_video_files"]
    )

    if actual_total_videos < minimum_video_files:
        raise ValueError(
            "Too few physical videos: "
            f"minimum {minimum_video_files}, "
            f"found {actual_total_videos}."
        )

    # --------------------------------------------------------------
    # Validate annotation categories and counts
    # --------------------------------------------------------------

    annotation_types = (
        dataframe[
            "video_annotation_type"
        ]
        .astype("string")
        .str.strip()
    )

    expected_annotation_types = {
        "partially_labelled",
        "fully_unlabelled",
    }

    invalid_annotation_mask = (
        annotation_types.isna()
        | annotation_types.eq("")
        | ~annotation_types.isin(
            expected_annotation_types
        )
    )

    if invalid_annotation_mask.any():
        invalid_annotation_types = (
            annotation_types.loc[
                invalid_annotation_mask
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            "Unexpected or missing video annotation "
            f"types: {invalid_annotation_types}"
        )

    annotation_counts = (
        annotation_types.value_counts()
    )

    actual_labelled_videos = int(
        annotation_counts.get(
            "partially_labelled",
            0,
        )
    )

    actual_unlabelled_videos = int(
        annotation_counts.get(
            "fully_unlabelled",
            0,
        )
    )

    expected_labelled_videos = len(frozenset(labelled_video_keys))
    expected_unlabelled_videos = actual_total_videos - expected_labelled_videos


    if (
        actual_labelled_videos
        != expected_labelled_videos
    ):
        raise ValueError(
            "Unexpected number of partially labelled "
            "videos: "
            f"expected {expected_labelled_videos}, "
            f"found {actual_labelled_videos}."
        )

    if (
        actual_unlabelled_videos
        != expected_unlabelled_videos
    ):
        raise ValueError(
            "Unexpected number of fully unlabelled "
            "videos: "
            f"expected {expected_unlabelled_videos}, "
            f"found {actual_unlabelled_videos}."
        )

    # --------------------------------------------------------------
    # Validate OpenCV probe results
    # --------------------------------------------------------------

    container_opened = (
        dataframe[
            "container_opened"
        ]
        .astype("boolean")
    )

    first_frame_readable = (
        dataframe[
            "first_frame_readable"
        ]
        .astype("boolean")
    )

    unreadable_container_mask = (
        container_opened.isna()
        | ~container_opened.fillna(False)
    )

    unreadable_first_frame_mask = (
        first_frame_readable.isna()
        | ~first_frame_readable.fillna(False)
    )

    if unreadable_container_mask.any():
        unreadable_videos = (
            dataframe.loc[
                unreadable_container_mask,
                "video_key",
            ]
            .tolist()
        )

        raise RuntimeError(
            "OpenCV could not open these videos: "
            f"{unreadable_videos}"
        )

    if unreadable_first_frame_mask.any():
        unreadable_videos = (
            dataframe.loc[
                unreadable_first_frame_mask,
                "video_key",
            ]
            .tolist()
        )

        raise RuntimeError(
            "OpenCV could not read the first frame of "
            f"these videos: {unreadable_videos}"
        )

    technical_metadata = (
        dataframe[
            [
                "container_fps",
                "container_reported_frame_count",
                "width",
                "height",
                "estimated_container_duration_seconds",
            ]
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    invalid_fps_mask = (
        technical_metadata[
            "container_fps"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "container_fps"
            ]
        )
        | technical_metadata[
            "container_fps"
        ].le(0)
    )

    invalid_frame_count_mask = (
        technical_metadata[
            "container_reported_frame_count"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "container_reported_frame_count"
            ]
        )
        | technical_metadata[
            "container_reported_frame_count"
        ].le(0)
        | technical_metadata[
            "container_reported_frame_count"
        ].mod(1).ne(0)
    )

    invalid_width_mask = (
        technical_metadata[
            "width"
        ].isna()
        | technical_metadata[
            "width"
        ].le(0)
        | technical_metadata[
            "width"
        ].mod(1).ne(0)
    )

    invalid_height_mask = (
        technical_metadata[
            "height"
        ].isna()
        | technical_metadata[
            "height"
        ].le(0)
        | technical_metadata[
            "height"
        ].mod(1).ne(0)
    )

    invalid_duration_mask = (
        technical_metadata[
            "estimated_container_duration_seconds"
        ].isna()
        | ~np.isfinite(
            technical_metadata[
                "estimated_container_duration_seconds"
            ]
        )
        | technical_metadata[
            "estimated_container_duration_seconds"
        ].le(0)
    )

    invalid_technical_mask = (
        invalid_fps_mask
        | invalid_frame_count_mask
        | invalid_width_mask
        | invalid_height_mask
        | invalid_duration_mask
    )

    if invalid_technical_mask.any():
        invalid_videos = (
            dataframe.loc[
                invalid_technical_mask,
                "video_key",
            ]
            .tolist()
        )

        raise ValueError(
            "Invalid technical video metadata found for: "
            f"{invalid_videos}"
        )

    # --------------------------------------------------------------
    # Validate expected container FPS
    # --------------------------------------------------------------

    expected_container_fps = float(
        config[
            "expected_export_container_fps"
        ]
    )

    fps_matches_expected = np.isclose(
        technical_metadata[
            "container_fps"
        ].to_numpy(
            dtype=float
        ),
        expected_container_fps,
        rtol=0.0,
        atol=1e-3,
    )

    if not fps_matches_expected.all():
        fps_mismatches = (
            dataframe.loc[
                ~fps_matches_expected,
                [
                    "video_key",
                    "container_fps",
                ],
            ]
            .to_dict(
                orient="records"
            )
        )

        raise ValueError(
            "Unexpected exported-container FPS values. "
            f"Expected {expected_container_fps}: "
            f"{fps_mismatches}"
        )

    # --------------------------------------------------------------
    # Validate total extractable frame inventory
    # --------------------------------------------------------------

    actual_total_frames = int(
        technical_metadata[
            "container_reported_frame_count"
        ].sum()
    )

    expected_total_frames = int(
        config[
            "expected_total_extractable_frames"
        ]
    )

    frame_difference = actual_total_frames - expected_total_frames

    frame_reference_status = (
        "matches_reference"
        if frame_difference == 0
        else "reference_mismatch"
    )

    if frame_reference_status == "reference_mismatch":
        warnings.warn(
            "Frame-count discrepancy against the published reference. "
            f"Reference: {expected_total_frames:,}; "
            f"container-reported: {actual_total_frames:,}; "
            f"difference: {frame_difference:+,}. "
            "The discrepancy remains unresolved.",
            UserWarning,
        )

    # --------------------------------------------------------------
    # Cross-check labelled metadata against physical videos
    # --------------------------------------------------------------

    manifest_labelled_video_keys = frozenset(
        dataframe.loc[
            annotation_types.eq(
                "partially_labelled"
            ),
            "video_key",
        ]
    )

    metadata_labelled_video_keys = frozenset(
        labelled_video_keys
    )

    missing_physical_videos = sorted(
        metadata_labelled_video_keys
        - manifest_labelled_video_keys
    )

    unexpected_labelled_videos = sorted(
        manifest_labelled_video_keys
        - metadata_labelled_video_keys
    )

    if (
        missing_physical_videos
        or unexpected_labelled_videos
    ):
        raise ValueError(
            "Video manifest is inconsistent with metadata. "
            "Missing physical labelled videos: "
            f"{missing_physical_videos}. "
            "Unexpected labelled videos: "
            f"{unexpected_labelled_videos}."
        )

    return dataframe


video_manifest = (
    video_manifest
    .pipe(
        validate_video_manifest,
        labelled_video_keys=(
            labelled_video_keys
        ),
        config=CONFIG,
    )
)



video_counts = (
    video_manifest[
        "video_annotation_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        "video_count"
    )
    .reset_index()
    .rename(
        columns={
            "video_annotation_type":
                "annotation_type"
        }
    )
)


video_inventory_summary = (
    pd.DataFrame.from_records(
        [
            {
                "total_videos":
                    len(video_manifest),

                "partially_labelled_videos":
                    int(
                        video_manifest[
                            "video_annotation_type"
                        ]
                        .eq(
                            "partially_labelled"
                        )
                        .sum()
                    ),

                "fully_unlabelled_videos":
                    int(
                        video_manifest[
                            "video_annotation_type"
                        ]
                        .eq(
                            "fully_unlabelled"
                        )
                        .sum()
                    ),

                "total_container_reported_frames":
                    int(
                        video_manifest[
                            "container_reported_frame_count"
                        ]
                        .sum()
                    ),

                "container_fps_min":
                    float(
                        video_manifest[
                            "container_fps"
                        ]
                        .min()
                    ),

                "container_fps_max":
                    float(
                        video_manifest[
                            "container_fps"
                        ]
                        .max()
                    ),
            }
        ]
    )
)


display(
    video_inventory_summary
)


display(
    video_counts
)

/tmp/ipykernel_1061/3961496634.py:496: UserWarning: Frame-count discrepancy against the published reference. Reference: 4,741,504; container-reported: 4,765,114; difference: +23,610. The discrepancy remains unresolved.
  warnings.warn(


,total_videos,partially_labelled_videos,fully_unlabelled_videos,total_container_reported_frames,container_fps_min,container_fps_max
0,117,43,74,4765114,30.0,30.00003


,annotation_type,video_count
0,fully_unlabelled,74
1,partially_labelled,43


### 11. Frame Alignment Audit

Annotated videos: 43; concurrent video workers: 43.
Decode: TorchCodec/beta on cuda:0, exact indexing, no CPU fallback.
SSIM: CUDA, per-video work streams, batches <= 8 pairs.
Offsets: (-2, -1, 0, 1, 2); transient batch-memory budget: 51.88 GiB.
CPU still handles file I/O, container indexing, reading reference PNG/JPEGs, and audit decisions.


All-video alignment candidates:   0%|          | 0/236190 [00:00<?, ?pair/s]

Video checkpoints:   0%|          | 0/43 [00:00<?, ?video/s]

Confirmed annotations: 0; excluded: 47,239.
Videos reused: 0; rescored: 43.
Audit: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results/frame_alignment_audit.csv
Confirmed Parquet: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/results/frame_alignment_confirmed.parquet


,filename,video_id,frame_number,finding_category,finding_class,x1,y1,x2,y2,x3,...,alignment_image_path,alignment_video_path,alignment_input_status,alignment_input_error,alignment_offset,aligned_frame_index,alignment_ssim,alignment_ssim_margin,alignment_status,alignment_training_eligible


### 12. Publish completed Sector 2 artifacts to GitHub

In [19]:

from pathlib import PurePosixPath

from urllib.parse import urlsplit
import pyarrow.parquet as pq

# Declarative publication policy. This does not replace runtime CONFIG.
SECTOR2_EXPORT = {
    "repo_url": GIT_CONFIG["repo_url"],
    "branch": GIT_CONFIG["branch"],
    "local_repo_dir": GIT_CONFIG["local_repo_dir"],
    "github_username": GIT_CONFIG["github_username"],
    "author_name": GIT_CONFIG["author_name"],
    "author_email": GIT_CONFIG["author_email"],
    "token_secret_name": GIT_CONFIG.get("token_secret_name", "GITHUB_TOKEN"),
    "repo_output_dir": "outputs/phase2/sector2",
    "confirm_push": True,                 # Show plan; type PUSH to approve.
    "push": True,                         # Commit + upload, not just local commit.
    "require_current_alignment_run": True,
    "include_physical_image_inventory": True,
    "gzip_csv_threshold_mib": 10,
    "always_gzip_csv": ["frame_alignment_candidates.csv"],
    "max_file_mib": 45,                    # After compression; fail, never silently skip.
    "max_export_mib": 250,
    "commit_message": "Phase 2 Sector 2: publish metadata and completed video/frame audits",
}

# ONLY these source families are published, not the whole Drive mount.
SECTOR2_FILES = [
    ("curated_data_dir", "dataset_audit/metadata_clean.parquet", "data/metadata_clean.parquet", True),
    ("manifests_dir", "video_manifest.csv", "manifests/video_manifest.csv", True),
    ("manifests_dir", "video_manifest_state.json", "manifests/video_manifest_state.json", True),
    ("manifests_dir", "video_inventory.csv", "manifests/video_inventory.csv", False),
    ("results_dir", "decode_audit_state.json", "results/decode_audit_state.json", True),
    ("results_dir", "decode_audit_run_state.json", "results/decode_audit_run_state.json", False),
    ("results_dir", "frame_alignment_state.json", "results/frame_alignment_state.json", True),
]
# All files listed in the two final output_sha256 dictionaries are added below.
# This includes confirmed Parquet, all candidate scores, exclusions and summaries.

_EXPORT_SCHEMA = "mmvqa-sector2-export-v1"
_COMMIT_MARKER = "MMVQA-Sector2-Export: v1"


def _s2_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _s2_json(value):
    return json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\n"


def _s2_atomic_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(dir=path.parent, suffix=".tmp")
    os.close(fd)
    temporary = Path(name)
    try:
        temporary.write_text(text, encoding="utf-8")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def _s2_relative(value):
    text = str(value)
    path = PurePosixPath(text)
    if (not text or path.is_absolute() or path == PurePosixPath(".")
            or ".." in path.parts or ".git" in path.parts
            or "\\" in text or any(ord(c) < 32 for c in text)):
        raise ValueError(f"Unsafe relative publication path: {text!r}")
    return path.as_posix()


def _s2_source(root, name):
    """Do not follow symlinks outside the declared artifact directories."""
    root = Path(root).resolve(strict=True)
    path = root / _s2_relative(name)
    walk = root
    for part in Path(name).parts:
        walk /= part
        if walk.is_symlink():
            raise ValueError(f"Symlink source is not publishable: {walk}")
    if not path.is_file() or not path.resolve().is_relative_to(root):
        raise FileNotFoundError(f"Required saved artifact is missing: {path}")
    return path


def _s2_state(path):
    state = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(state, dict) or state.get("complete") is not True:
        raise RuntimeError(f"The audit is not complete: {path}. Finish/rerun its audit cell first.")
    return state


def _s2_validate_states(config, dirs, settings):
    """Validate committed reports; does not revalidate or read raw video bytes."""
    run_id = str(config.get("run_id", ""))
    if not re.fullmatch(r"[A-Za-z0-9_-]+", run_id):
        raise ValueError("CONFIG['run_id'] is missing or unsafe. Do not regenerate it here.")
    results = Path(dirs["results_dir"])
    manifests = Path(dirs["manifests_dir"])
    states = {
        "manifest": _s2_state(_s2_source(manifests, "video_manifest_state.json")),
        "decode": _s2_state(_s2_source(results, "decode_audit_state.json")),
        "alignment": _s2_state(_s2_source(results, "frame_alignment_state.json")),
    }
    expected = {str(_s2_source(manifests, "video_manifest.csv")): states["manifest"]["csv_sha256"]}
    required = {
        "decode": {"decode_audit.csv", "decode_audit_summary.csv", "labelled_video_decode_audit.csv", "unlabelled_video_decode_audit.csv"},
        "alignment": {"frame_alignment_audit.csv", "frame_alignment_candidates.csv", "frame_alignment_offset_summary.csv", "frame_alignment_confirmed.csv", "frame_alignment_excluded.csv", "frame_alignment_confirmed.parquet", "frame_alignment_performance.csv"},
    }
    for family in ("decode", "alignment"):
        hashes = states[family].get("output_sha256")
        if not isinstance(hashes, dict) or not required[family].issubset(hashes):
            raise ValueError(f"Incomplete {family} output_sha256 manifest.")
        for name, digest in hashes.items():
            if Path(name).name != name or Path(name).suffix not in {".csv", ".json", ".parquet"}:
                raise ValueError(f"Unexpected audit output filename: {name!r}")
            path = _s2_source(results, name)
            expected[str(path)] = digest
    for name, digest in expected.items():
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest) or _s2_sha(name) != digest:
            raise ValueError(f"Saved audit checksum mismatch: {name}. Do not publish stale/edited reports.")
    alignment = states["alignment"]
    if settings["require_current_alignment_run"] and alignment.get("run_id") != run_id:
        raise ValueError("Alignment belongs to a different run_id. Rerun alignment (valid caches are reusable) before publication.")
    effective = alignment.get("effective_settings", {})
    for recorded, configured in (("min_ssim", "frame_alignment_min_ssim"), ("min_margin", "frame_alignment_min_margin")):
        if effective.get(recorded) != config.get(configured):
            raise ValueError(f"Current CONFIG differs from completed alignment: {configured}.")
    if sorted(effective.get("candidate_offsets", [])) != sorted(config.get("frame_index_offset_candidates", [])):
        raise ValueError("Current candidate offsets differ from completed alignment.")
    if (results / "decode_audit_run_state.json").exists():
        _s2_state(_s2_source(results, "decode_audit_run_state.json"))
    # Snapshot state bytes as well, to detect changes during review/copying.
    for parent, name in ((manifests, "video_manifest_state.json"), (results, "decode_audit_state.json"), (results, "frame_alignment_state.json")):
        expected[str(_s2_source(parent, name))] = _s2_sha(parent / name)
    return states, expected


def _s2_summary(config, dirs, states):
    """Small summaries derived from the verified saved tables, not stale globals."""
    results = Path(dirs["results_dir"])
    manifest = pd.read_csv(Path(dirs["manifests_dir"]) / "video_manifest.csv", dtype="string", keep_default_na=False)
    decode = pd.read_csv(results / "decode_audit.csv", dtype="string", keep_default_na=False)
    metadata = Path(dirs["curated_data_dir"]) / "dataset_audit/metadata_clean.parquet"
    original_rows = pq.ParquetFile(metadata).metadata.num_rows
    confirmed = pd.read_parquet(results / "frame_alignment_confirmed.parquet")
    align = states["alignment"]
    if original_rows != align.get("original_annotation_rows") or len(confirmed) != align.get("confirmed_annotation_rows"):
        raise ValueError("Parquet row counts disagree with the completed alignment state.")
    excluded = pd.read_csv(results / "frame_alignment_excluded.csv", usecols=["video_key"], dtype="string")
    if len(excluded) != align.get("excluded_annotation_rows") or len(confirmed) + len(excluded) != original_rows:
        raise ValueError("Confirmed/excluded subsets do not partition the original metadata rows.")
    keys = manifest["video_key"]
    if keys.duplicated().any() or decode["video_key"].duplicated().any() or set(keys) != set(decode["video_key"]):
        raise ValueError("Manifest and decode audit must contain the same unique videos.")
    if len(manifest) != states["manifest"].get("video_count") or len(decode) != states["decode"].get("video_count"):
        raise ValueError("Video counts disagree with the state files.")
    if not confirmed.empty:
        if not {"video_key", "aligned_frame_index", "alignment_training_eligible"}.issubset(confirmed):
            raise ValueError("Confirmed Parquet lacks downstream alignment columns.")
        if not confirmed["alignment_training_eligible"].astype("string").str.lower().eq("true").all():
            raise ValueError("Confirmed Parquet contains a row not accepted by alignment.")
        if not set(confirmed["video_key"].astype(str)).issubset(set(keys)):
            raise ValueError("Confirmed annotations refer to unknown videos.")
        indices = pd.to_numeric(confirmed["aligned_frame_index"], errors="raise")
        if indices.isna().any() or indices.lt(0).any() or indices.mod(1).ne(0).any():
            raise ValueError("Invalid zero-based frame indices in confirmed metadata.")
    audit = pd.read_csv(results / "frame_alignment_audit.csv", usecols=["alignment_status"], dtype="string")
    decode_summary = pd.read_csv(results / "decode_audit_summary.csv")
    return {
        "schema": _EXPORT_SCHEMA, "run_id": str(config["run_id"]),
        "alignment_created_utc": align.get("created_utc"),
        "original_annotation_rows": original_rows,
        "confirmed_annotation_rows": len(confirmed), "excluded_annotation_rows": len(excluded),
        "unique_alignment_mappings": align.get("unique_mappings"),
        "evaluated_offsets": align.get("evaluated_offsets"),
        "alignment_status_counts": {str(k): int(v) for k, v in audit["alignment_status"].value_counts().items()},
        "video_count": len(manifest),
        "total_container_reported_frames": int(pd.to_numeric(manifest["container_reported_frame_count"]).sum()),
        "decode_summary": json.loads(decode_summary.to_json(orient="records")),
        "all_videos_decoded": states["decode"].get("all_videos_decoded"),
        "videos_needing_review": states["decode"].get("videos_needing_review"),
        "alignment_engine": align.get("engine"),
        "empty_confirmed_subset": confirmed.empty,
        "interpretation": "Completed technical audit, not a medical validation. Published-reference discrepancies and exclusions are retained. Source videos/images and decoder caches are not included.",
    }


def _s2_no_secrets(config):
    """CONFIG contains values, not tokens; secret-name references are permitted."""
    def visit(value, trail="CONFIG"):
        if isinstance(value, dict):
            for key, item in value.items():
                key = str(key)
                if re.search(r"(?i)(token|password|api_key|private_key|authorization|access_key|client_secret)", key) and not key.endswith(("_name", "_secret_name")) and item:
                    raise ValueError(f"Possible credential field {trail}.{key}: keep credentials in Colab Secrets.")
                visit(item, f"{trail}.{key}")
        elif isinstance(value, (tuple, list)):
            for item in value:
                visit(item, trail)
    visit(config)


def _s2_scan_bytes(path, token=""):
    # Conservative convenience check; not a substitute for reviewing public data.
    pattern = re.compile(rb"(?:github_pat_[A-Za-z0-9_]{20,}|gh[pousr]_[A-Za-z0-9]{20,}|-----BEGIN [A-Z ]*PRIVATE KEY-----)")
    tail = b""
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value = tail + block
            if pattern.search(value) or (token and token.encode() in value):
                raise ValueError(f"Possible credential in selected artifact: {Path(path).name}")
            tail = value[-4096:]


def _s2_prepare_bundle(config, dirs, settings, staging, token=""):
    _s2_no_secrets(config)
    states, expected = _s2_validate_states(config, dirs, settings)
    spec = list(SECTOR2_FILES)
    if settings["include_physical_image_inventory"]:
        spec.append(("curated_data_dir", "dataset_audit/physical_image_inventory.csv", "data/physical_image_inventory.csv", True))
    for family in ("decode", "alignment"):
        spec += [("results_dir", name, "results/" + name, True) for name in states[family]["output_sha256"]]
    # Other existing top-level summary JSONs only; never recursively include caches.
    for path in sorted(Path(dirs["results_dir"]).glob("*summary*.json")):
        spec.append(("results_dir", path.name, "results/" + path.name, False))
    summary = _s2_summary(config, dirs, states)
    artifacts, watch, seen, skipped = [], dict(expected), set(), []
    root = Path(config["storage_root"]).resolve(strict=True)
    for directory, relative, target, required in spec:
        target = _s2_relative(target)
        if target in seen:
            continue
        seen.add(target)
        if not (Path(dirs[directory]) / relative).exists() and not required:
            skipped.append(target)
            continue
        source = _s2_source(dirs[directory], relative)
        if not source.resolve().is_relative_to(root):
            raise ValueError(f"Artifact lies outside CONFIG storage_root: {source}")
        digest = _s2_sha(source)
        if str(source) in expected and expected[str(source)] != digest:
            raise RuntimeError(f"An audit changed after validation: {source}")
        watch[str(source)] = digest
        _s2_scan_bytes(source, token)
        size = source.stat().st_size
        compressed = source.suffix == ".csv" and (source.name in settings["always_gzip_csv"] or size >= settings["gzip_csv_threshold_mib"] * 1024**2)
        exported = target + ".gz" if compressed else target
        dest = Path(staging) / exported
        dest.parent.mkdir(parents=True, exist_ok=True)
        if compressed:
            # No time or filename in gzip header: reruns produce the same bytes.
            with source.open("rb") as inp, dest.open("wb") as out:
                with gzip.GzipFile(filename="", fileobj=out, mode="wb", compresslevel=6, mtime=0) as zipped:
                    shutil.copyfileobj(inp, zipped)
            with gzip.open(dest, "rb") as stream:
                decoded = hashlib.sha256()
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    decoded.update(block)
            if decoded.hexdigest() != digest:
                raise RuntimeError(f"Gzip verification failed: {source}")
        else:
            shutil.copyfile(source, dest)
            if _s2_sha(dest) != digest:
                raise RuntimeError(f"Copy changed while being made: {source}")
        if _s2_sha(source) != digest:
            raise RuntimeError(f"Source changed during export: {source}")
        artifacts.append({
            "logical_path": target, "path": exported,
            "source_storage_relative": source.relative_to(root).as_posix(),
            "source_bytes": size, "source_sha256": digest,
            "bytes": dest.stat().st_size, "sha256": _s2_sha(dest),
            "encoding": "gzip" if compressed else "identity",
        })
    config_name = f"phase2_02_video_frame_validation_{config['run_id']}_config.json"
    generated = {
        "configs/" + config_name: _s2_json(config),
        "configs/github_export_settings.json": _s2_json(settings),
        "reports/sector2_summary.json": _s2_json(summary),
        "reports/sector2_summary.md": (
            "# Phase 2 — Sector 2: published artifact summary\n\n"
            f"Run: `{config['run_id']}`\n\n"
            "| Measurement | Value |\n|---|---:|\n"
            f"| Videos | {summary['video_count']:,} |\n"
            f"| Original annotation rows | {summary['original_annotation_rows']:,} |\n"
            f"| Confirmed annotation rows | {summary['confirmed_annotation_rows']:,} |\n"
            f"| Excluded annotation rows | {summary['excluded_annotation_rows']:,} |\n"
            f"| Container-reported frames | {summary['total_container_reported_frames']:,} |\n\n"
            "See `sector2_summary.json` and `decode_audit_summary.csv` for decoded counts and published-reference differences.\n\n"
            "Confirmed means accepted by configured SSIM/ambiguity rules, not medical validation. "
            "Exclusions and an empty confirmed subset are reported, never silently repaired.\n"
        ),
        "README.md": _S2_BUNDLE_README,
    }
    for relative, text in generated.items():
        path = Path(staging) / relative
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        _s2_scan_bytes(path, token)
        artifacts.append({"logical_path": relative, "path": relative, "source_storage_relative": None,
                          "source_bytes": None, "source_sha256": None,
                          "bytes": path.stat().st_size, "sha256": _s2_sha(path), "encoding": "identity"})
    artifacts.sort(key=lambda row: row["path"])
    catalog = {
        "schema": _EXPORT_SCHEMA, "run_id": str(config["run_id"]),
        "alignment_created_utc": states["alignment"].get("created_utc"),
        "config_file": "configs/" + config_name,
        "artifacts": artifacts,
        "excluded_families": ["raw videos", "raw images", "temporary copies", "per-video caches", "partial reports", "legacy backups", "notebook outputs/credentials"],
        "optional_files_not_found": skipped,
        "notes": "CSV gzip hashes differ from originals. Validate original audit SHA-256 after decompression. This catalog describes the current export; earlier config snapshots may remain in Git history/working tree.",
    }
    (Path(staging) / "artifact_catalog.json").write_text(_s2_json(catalog), encoding="utf-8")
    files = sorted(p for p in Path(staging).rglob("*") if p.is_file())
    for path in files:
        if path.stat().st_size > settings["max_file_mib"] * 1024**2:
            raise ValueError(f"Artifact exceeds {settings['max_file_mib']} MiB even after compression: {path.name}. Use LFS/Releases or a deliberate partition; no artifact was silently skipped.")
    if sum(p.stat().st_size for p in files) > settings["max_export_mib"] * 1024**2:
        raise ValueError("Total bundle exceeds the configured publication budget.")
    return catalog, watch, summary


_S2_BUNDLE_README = '''# Phase 2 — Sector 2 artifacts

This folder contains completed video/frame audit outputs and downstream metadata.
It does not contain the raw Kvasir-Capsule images/videos, model weights or runtime caches.
The current artifact set, file hashes and optional gzip encodings are listed in `artifact_catalog.json`.
Git history versions the data; timestamps do not duplicate entire result folders.

## Downstream inputs

- `data/metadata_clean.parquet`: original cleaned Sector 1 annotations; not replaced by filtering.
- `data/physical_image_inventory.csv` (possibly `.gz`): Sector 1 image-path lookup.
- `results/frame_alignment_confirmed.parquet`: aligned subset, including zero-based `aligned_frame_index`.
- `manifests/video_manifest.csv`: source video identifiers and container probe metadata.
- `results/decode_audit.csv`: observed decoded counts and decoder provenance.
- `results/frame_alignment_audit.csv`, candidates, exclusions, offset summary and state JSON: all decisions and their evidence.

```python
from pathlib import Path
import json, pandas as pd

bundle = Path("outputs/phase2/sector2")  # relative to your repository root
catalog = json.loads((bundle / "artifact_catalog.json").read_text())
paths = {r["logical_path"]: bundle / r["path"] for r in catalog["artifacts"]}
df_clean = pd.read_parquet(paths["data/metadata_clean.parquet"])
df_alignment_confirmed = pd.read_parquet(paths["results/frame_alignment_confirmed.parquet"])
video_manifest = pd.read_csv(paths["manifests/video_manifest.csv"], dtype={"video_key": "string"})
candidates = pd.read_csv(paths["results/frame_alignment_candidates.csv"], dtype={"video_key": "string"})
```

Pandas reads `.csv.gz` automatically. The data are not all-to-all matches: each image
is tested only against candidate frames in its own video. Acceptance is a technical
filter under the documented rules, not proof that excluded labels are medically wrong.

## Paths and provenance

Source absolute paths are preserved as audit evidence; they are NOT portable by themselves.
In `video_manifest.csv`, `video_relpath` is relative to CONFIG's `storage_root`.
In `decode_audit.csv`, `video_relpath` is relative to `DIRS["dataset_root_dir"]`.
In the Sector 1 image inventory, `relative_path` is relative to `DIRS["raw_data_dir"]`.
To read actual pixels in another runtime, mount/download the original dataset and rebase
these paths. GitHub alone does not provide raw videos or images.
For confirmed `alignment_image_path` / `alignment_video_path`, replace the saved
CONFIG storage-root prefix with your new storage root, retaining the remaining path.

State JSONs are copied unchanged. Their output hashes refer to ORIGINAL report bytes.
For gzipped reports, decompress before comparing to those hashes; the catalog also
stores the hash of the compressed Git artifact. Do not copy these state JSONs back as
working decoder caches: the per-video caches remain on Drive.

## Source attribution

Kvasir-Capsule dataset: https://datasets.simula.no/kvasir-capsule/
Dataset paper: https://doi.org/10.1038/s41597-021-00920-z
These exports derive from the dataset and this project's processing, not a new official
dataset release. Preserve source attribution and applicable source terms when reusing them.
'''


def _s2_repo_identity(url):
    parsed = urlsplit(str(url))
    if (parsed.scheme != "https" or parsed.hostname != "github.com" or parsed.username or parsed.password
            or parsed.port not in (None, 443) or parsed.query or parsed.fragment):
        raise ValueError("Use an HTTPS github.com repository URL WITHOUT embedded credentials.")
    path = parsed.path.rstrip("/").removesuffix(".git")
    if not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", path):
        raise ValueError("Unexpected GitHub repository path.")
    return "https://github.com" + path


@contextmanager
def _s2_auth(settings):
    """Token goes to an ephemeral child environment, never URL/arguments/disk."""
    token = os.environ.get(settings["token_secret_name"], "").strip()
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get(settings["token_secret_name"]).strip()
        except Exception as error:
            if settings["push"]:
                raise RuntimeError(f"Enable notebook access to Colab Secret {settings['token_secret_name']!r}. No commit was created.") from None
    if settings["push"] and not token:
        raise ValueError("A token is required to push. Keep it in Colab Secrets, not CONFIG.")
    with tempfile.TemporaryDirectory(prefix="sector2-auth-") as folder:
        askpass = Path(folder) / "askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" "$MMVQA_EXPORT_USER" ;;\n*) printf "%s\\n" "$MMVQA_EXPORT_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env = {k: v for k, v in os.environ.items() if not k.startswith("GIT_")}
        env.update(GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(askpass), LC_ALL="C",
                   MMVQA_EXPORT_USER=settings["github_username"], MMVQA_EXPORT_TOKEN=token)
        yield env, token


def _s2_git(repo, *args, env=None, input_text=None, accepted=(0,)):
    command = ["git", "-c", "credential.helper="]
    if repo is not None:
        command += ["-C", str(repo)]
    result = subprocess.run(command + list(args), input=input_text, text=True,
                            capture_output=True, env=env, timeout=600)
    if result.returncode not in accepted:
        message = (result.stderr or result.stdout).strip()
        token = (env or {}).get("MMVQA_EXPORT_TOKEN", "")
        if token:
            message = message.replace(token, "[REDACTED]")
        raise RuntimeError(f"Git operation failed (no automatic reset/rebase/force-push):\n{message}")
    return result.stdout


def _s2_clean(repo, env):
    if _s2_git(repo, "status", "--porcelain", "--untracked-files=all", env=env).strip():
        raise RuntimeError("The checkout has local changes. Save/commit or move them deliberately before publishing. This cell will not stash, reset or include unrelated files.")


def _s2_refresh(settings, env):
    repo = Path(settings["local_repo_dir"]).expanduser().resolve()
    branch = settings["branch"]
    _s2_git(None, "check-ref-format", "--branch", branch, env=env)
    expected = _s2_repo_identity(settings["repo_url"])
    if not (repo / ".git").exists():
        if repo.exists() and (not repo.is_dir() or any(repo.iterdir())):
            raise ValueError("local_repo_dir exists but is not an empty Git checkout.")
        repo.parent.mkdir(parents=True, exist_ok=True)
        _s2_git(None, "clone", "--branch", branch, "--single-branch", settings["repo_url"], str(repo), env=env)
    for push_options in ((), ("--push",)):
        urls = _s2_git(repo, "remote", "get-url", *push_options, "--all", "origin", env=env).splitlines()
        if len(urls) != 1 or _s2_repo_identity(urls[0]) != expected:
            raise ValueError("Origin/push destination differs from the configured repository.")
    if _s2_git(repo, "branch", "--show-current", env=env).strip() != branch:
        raise RuntimeError("Wrong checkout branch; no automatic branch switch is performed.")
    _s2_clean(repo, env)
    print(f"Fetching origin/{branch} before copying artifacts...")
    remote = f"refs/remotes/origin/{branch}"
    _s2_git(repo, "fetch", "--no-tags", "origin", f"refs/heads/{branch}:{remote}", env=env)
    ahead, behind = map(int, _s2_git(repo, "rev-list", "--left-right", "--count", f"HEAD...{remote}", env=env).split())
    if ahead and behind:
        raise RuntimeError("Local/remote branches diverged. Review and integrate them manually; no history is rewritten.")
    if behind:
        _s2_git(repo, "merge", "--ff-only", remote, env=env)
    if ahead:
        # Retry a prior failed push ONLY for commits made by this exporter.
        prefix = _s2_relative(settings["repo_output_dir"]) + "/"
        for commit in _s2_git(repo, "rev-list", f"{remote}..HEAD", env=env).splitlines():
            parents = _s2_git(repo, "show", "-s", "--format=%P", commit, env=env).split()
            message = _s2_git(repo, "show", "-s", "--format=%B", commit, env=env)
            names = _s2_git(repo, "diff-tree", "--no-commit-id", "--name-only", "-r", "-z", commit, env=env).split("\0")
            if len(parents) != 1 or _COMMIT_MARKER not in message.splitlines() or any(n and not n.startswith(prefix) for n in names):
                raise RuntimeError("Unpublished commits not owned by this exporter exist. Review them before publication.")
    return repo, ahead


def publish_sector2_github(config, dirs, settings):
    """Make one scoped, checksum-verified commit and normally push it to GitHub."""
    prefix = _s2_relative(settings["repo_output_dir"])
    if not str(settings["author_name"]).strip() or "@" not in str(settings["author_email"]):
        raise ValueError("Set author_name and author_email in GIT_CONFIG.")
    with _s2_auth(settings) as (env, token), tempfile.TemporaryDirectory(prefix="sector2-publish-") as work:
        staging = Path(work)
        catalog, sources, summary = _s2_prepare_bundle(config, dirs, settings, staging, token)
        files = sorted(p for p in staging.rglob("*") if p.is_file())
        repo, ahead = _s2_refresh(settings, env)
        records = []
        for source in files:
            relative = prefix + "/" + source.relative_to(staging).as_posix()
            target = repo / relative
            walk = repo
            for part in Path(relative).parts:
                walk /= part
                if walk.is_symlink():
                    raise ValueError(f"Symlink Git destination: {walk}")
            if not target.resolve().is_relative_to(repo):
                raise ValueError("Unsafe Git destination.")
            changed = not target.is_file() or _s2_sha(target) != _s2_sha(source)
            if changed:
                records.append((source, target, relative))
        print(f"Selected: {len(files)} files; changed: {len(records)}; bundle: {sum(p.stat().st_size for p in files)/1024**2:.2f} MiB")
        print(f"Confirmed metadata rows: {summary['confirmed_annotation_rows']:,}; excluded: {summary['excluded_annotation_rows']:,}")
        if summary["empty_confirmed_subset"]:
            print("WARNING: confirmed subset is empty. Reports can be published, but this is not a training-ready dataset.")
        for source, _, relative in records:
            print(f"  {source.stat().st_size / 1024**2:7.2f} MiB  {relative}")
        if (records or ahead) and settings["confirm_push"]:
            print("Selected artifacts will be visible to everyone with access to this GitHub repository.")
            action = "Commit + push" if settings["push"] else "Commit locally"
            if input(f"{action} this selection? Type PUSH: ").strip() != "PUSH":
                return {"status": "cancelled", "changed_files": len(records)}
        # Source reports must stay unchanged throughout review/staging.
        for name, digest in sources.items():
            if _s2_sha(name) != digest:
                raise RuntimeError(f"Source changed during review: {name}. Rerun export.")
        _s2_clean(repo, env)
        if records:
            for source, target, _ in records:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source, target)
            paths = [relative for _, _, relative in records]
            # --force here ONLY bypasses .gitignore for this exact approved list.
            # There is never a force PUSH or a 'git add .' in this cell.
            _s2_git(repo, "--literal-pathspecs", "add", "--force", "--pathspec-from-file=-", "--pathspec-file-nul",
                    env=env, input_text="\0".join(paths) + "\0")
            staged = set(filter(None, _s2_git(repo, "diff", "--cached", "--name-only", "-z", env=env).split("\0")))
            if staged - set(paths):
                raise RuntimeError("Unexpected staged changes; nothing was committed/pushed.")
            for source, _, relative in records:
                expected = _s2_git(repo, "hash-object", "--no-filters", str(source), env=env).strip()
                actual = _s2_git(repo, "rev-parse", ":" + relative, env=env).strip()
                if actual != expected:
                    raise RuntimeError(f"Git filters changed bytes for {relative}. Check .gitattributes/LFS; no commit was created.")
            if staged:
                print(_s2_git(repo, "diff", "--cached", "--stat", env=env))
                message = settings["commit_message"] + f"\n\nRun-ID: {config['run_id']}\n" + _COMMIT_MARKER
                _s2_git(repo, "-c", f"user.name={settings['author_name']}", "-c", f"user.email={settings['author_email']}", "commit", "-m", message, env=env)
        head = _s2_git(repo, "rev-parse", "HEAD", env=env).strip()
        remote = f"refs/remotes/origin/{settings['branch']}"
        pending = int(_s2_git(repo, "rev-list", "--count", f"{remote}..HEAD", env=env))
        status = "unchanged"
        if pending and settings["push"]:
            try:
                _s2_git(repo, "push", "origin", f"HEAD:refs/heads/{settings['branch']}", env=env)
            except RuntimeError as error:
                raise RuntimeError(f"Local commit {head} is preserved; push did NOT succeed. Check token permissions/branch protection or remote changes. No force-push was attempted.\n{error}") from None
            status = "pushed"
        elif pending:
            status = "committed_locally"
        receipt = {
            "status": status, "run_id": str(config["run_id"]), "commit": head,
            "repository": _s2_repo_identity(settings["repo_url"]), "branch": settings["branch"],
            "repo_output_dir": prefix, "artifact_catalog_sha256": _s2_sha(staging / "artifact_catalog.json"),
            "selected_files": len(files), "changed_files": len(records),
            "recorded_utc": datetime.now(timezone.utc).isoformat(),
        }
        receipt_path = Path(dirs["reports_dir"]) / f"phase2_02_github_publish_{config['run_id']}.json"
        _s2_atomic_text(receipt_path, _s2_json(receipt))
        # Keep a catalog copy on Drive; do not include this receipt in future bundles.
        _s2_atomic_text(Path(dirs["reports_dir"]) / "sector2_github_artifact_catalog.json", _s2_json(catalog))
        _s2_atomic_text(Path(dirs["reports_dir"]) / "sector2_summary.json", _s2_json(summary))
        print(f"Publication status: {status}; commit: {head}")
        print("GitHub:", receipt["repository"] + "/tree/" + head + "/" + prefix)
        print("Drive receipt:", receipt_path)
        return receipt


# Execute after ALL processing cells. CONFIG and raw data are not overwritten.
sector2_github_publication = publish_sector2_github(CONFIG, DIRS, SECTOR2_EXPORT)


Fetching origin/main before copying artifacts...
Selected: 25 files; changed: 25; bundle: 9.02 MiB
Confirmed metadata rows: 0; excluded: 47,239
     0.00 MiB  outputs/phase2/sector2/README.md
     0.01 MiB  outputs/phase2/sector2/artifact_catalog.json
     0.00 MiB  outputs/phase2/sector2/configs/github_export_settings.json
     0.00 MiB  outputs/phase2/sector2/configs/phase2_02_video_frame_validation_20260925_211052_config.json
     0.91 MiB  outputs/phase2/sector2/data/metadata_clean.parquet
     0.59 MiB  outputs/phase2/sector2/data/physical_image_inventory.csv.gz
     0.01 MiB  outputs/phase2/sector2/manifests/video_inventory.csv
     0.03 MiB  outputs/phase2/sector2/manifests/video_manifest.csv
     0.00 MiB  outputs/phase2/sector2/manifests/video_manifest_state.json
     0.00 MiB  outputs/phase2/sector2/reports/sector2_summary.json
     0.00 MiB  outputs/phase2/sector2/reports/sector2_summary.md
     0.06 MiB  outputs/phase2/sector2/results/decode_audit.csv
     0.00 MiB  outputs